# NimbusCompute — Consumption Revenue & Cash Collections Forecast: June–August 2026


---
## Block 0 — Task and solution plan

### Context

NimbusCompute is a B2B cloud infrastructure provider. The company sells compute resources (compute, GPU), object storage, and network egress to enterprise customers across the SaaS, fintech, AI, gaming, and enterprise segments.

Finance and Sales leadership need a forecast for **June, July, and August 2026** across two metrics:

1. **Consumption revenue** — expected revenue from customer cloud consumption, computed from usage volumes, the price list, and contract discounts.
2. **Cash collections** — expected cash receipts, accounting for invoice timing, payment terms, contract structures, and the state of accounts receivable.

The forecast requires three scenarios: **Base**, **Upside**, **Downside**.

---

### Data

Nine tables from the company's operating systems are available for analysis:

| Table | Contents |
|---|---|
| `billing_customers` | Billing-system customer base: segment, status, payment terms |
| `monthly_usage` | Historical usage volumes by product and customer — through May 2026 inclusive |
| `price_list` | Price list with product price-change history |
| `contracts` | Contracts: commitment volume, discounts, billing type, terms |
| `invoices` | Invoices: amounts, issue dates, due and paid dates, statuses |
| `crm_accounts` | CRM accounts linked to billing customers |
| `crm_opportunities` | Sales pipeline: stage, probability, expected dates and amounts |
| `crm_opportunity_products` | Product mix and revenue estimates within each opportunity |
| `business_events` | Business context affecting interpretation of historical data |

---

### Expected deliverables

Per the assignment, we need to prepare:

- SQL queries as the primary analytical tool
- A reproducible Jupyter notebook with full code
- A CSV file with the consumption revenue and cash collections forecast by month and scenario
- A short executive summary for leadership
- Additional charts and tables as needed

---

### Solution plan

The analysis is structured into six sequential blocks:

**Block 1 — Data review.**
Assessing the quality of the source data: anomalies, business events, customer and invoice statuses. Deciding which data and periods enter the calculation base and which require adjustment or exclusion.

**Block 2 — Historical revenue.**
Computing actual consumption revenue over the available historical period. Analyzing dynamics, product and customer structure. Forming the baseline for the forecast.

**Block 3 — Consumption forecast.**
Projecting the existing customer base forward given observed trends. Adding the expected pipeline contribution. Building three scenarios.

**Block 4 — Cash forecast.**
Forecasting actual collections given billing lag, customer payment behavior, contract types, and the state of receivables.

**Block 5 — Summary and export.**
Final tables across the three scenarios, export of `forecast_output.csv`, final visualizations.

**Block 6 — Executive summary.**
Key takeaways for Finance and Sales: expected ranges, the gap between consumption and cash, key drivers and risks.


In [1]:
import pandas as pd
from db import get_con
import plotly.express as px

con = get_con()

# ── as-of forecast cutoff date: the last invoice_date in the extract ──
AS_OF = pd.Timestamp("2026-06-06")

# Single cleaned invoice layer: dedup by invoice_id, excluding test_invoice and void_pending.
# All cash/receivables blocks use it (not the raw invoices table).
con.sql("""
    CREATE OR REPLACE TEMP VIEW clean_invoices AS
    SELECT * EXCLUDE (rn) FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY invoice_id ORDER BY invoice_date DESC) AS rn
        FROM invoices
        WHERE invoice_type <> 'test_invoice'
          AND payment_status <> 'void_pending'
    ) WHERE rn = 1
""")

# Receivables open EXACTLY as of the cutoff date (not the final 'open' status from the extract):
# invoice issued on or before as-of and not yet paid as of as-of. Includes invoices formally
# marked 'paid' but with paid_date after the cutoff — as of the forecast date they were AR.
con.sql(f"""
    CREATE OR REPLACE TEMP VIEW open_ar_asof AS
    SELECT * FROM clean_invoices
    WHERE invoice_type = 'usage'
      AND invoice_date <= DATE '{AS_OF.date()}'
      AND NOT (paid_date IS NOT NULL AND paid_date <= DATE '{AS_OF.date()}')
""")

# layer validation
_ci = con.sql("SELECT * FROM clean_invoices").df()
assert _ci["invoice_id"].is_unique, "clean_invoices: invoice_id is not unique"
assert not (_ci["invoice_type"] == "test_invoice").any()
# guard the date-range discount join against overlapping active contracts (else usage double counts)
_ov = con.sql("""
    SELECT COUNT(*) n FROM contracts a JOIN contracts b
      ON a.customer_id=b.customer_id AND a.contract_id<b.contract_id
     AND a.contract_start<=b.contract_end AND b.contract_start<=a.contract_end
""").df()["n"].iloc[0]
assert _ov == 0, f"Overlapping contracts ({_ov}) — the discount join may double count revenue"
_ar = con.sql("SELECT ROUND(SUM(amount_usd)/1000,1) k FROM open_ar_asof").df()["k"].iloc[0]
print(f"clean_invoices: {len(_ci)} invoices | AS_OF = {AS_OF.date()} | AR as-of = {_ar}K")


Setting up database tables...
Done. All tables are ready.
clean_invoices: 2160 invoices | AS_OF = 2026-06-06 | AR as-of = 4011.0K


---
## Block 1 — Data review

Before building the forecast — a detailed review of each table: coverage by time and customer, price changes, usage anomalies, the state of contracts, invoices, and the sales pipeline. Every finding is recorded as a flag for the calculation blocks.


### 1.0 Business events — context for interpreting the data


In [2]:
df_events = con.sql("""
    SELECT event_date, event_type, affected_product, affected_customer_id, description
    FROM business_events
    ORDER BY event_date
""").df()
df_events

,event_date,event_type,affected_product,affected_customer_id,description
0,2025-07-01,pricing_change,compute_standard,None,Compute list price reduced for new and renewin...
1,2025-10-01,product_migration,gpu_a100,None,Legacy GPU SKU migrated to gpu_accelerated in ...
2,2026-03-15,capacity_constraint,gpu_accelerated,None,Temporary GPU supply constraint delayed some e...
3,2026-04-04,billing_system_change,NaN,None,Invoice export switched to a new source table;...
4,2026-05-20,sales_policy,NaN,None,Sales leadership asked teams to split saving-p...


**Findings:**

Five business events are recorded, all in 2025–2026, and all directly affect data interpretation:

- **pricing_change (July 2025)** — a price change for compute_standard. Historical revenue cannot be computed with a single price.
- **product_migration (October 2025)** — SKU `gpu_a100` replaced by `gpu_accelerated`. This is one product under two names; they must be combined into a single time series.
- **capacity_constraint (March 2026)** — a temporary GPU capacity shortage. Usage that month may be understated for some customers — an anomaly that does not reflect real demand.
- **billing_system_change (April 2026)** — a change in the invoice data source. Artifacts are possible in the April data.
- **sales_policy (May 2026)** — large deals started being split into parts. CRM may contain duplicates.

> **📋 Calculation note:** GPU baseline — April–May 2026, exclude March. Apply prices by date range. In the pipeline, check for same-account deals with close expected close dates.


### 1.1 Price list — price change history


In [3]:
df_prices = con.sql("""
    SELECT product, effective_from, unit, list_price_usd, notes
    FROM price_list
    ORDER BY product, effective_from
""").df()
df_prices

,product,effective_from,unit,list_price_usd,notes
0,compute_standard,2024-01-01,compute_hour,0.046,General purpose VM compute.
1,compute_standard,2025-07-01,compute_hour,0.043,Mid-2025 price decrease for compute.
2,gpu_a100,2024-01-01,gpu_hour,2.650,Legacy GPU SKU used before product migration.
3,gpu_accelerated,2025-10-01,gpu_hour,2.550,Renamed GPU SKU after platform migration.
4,network_egress,2024-01-01,egress_tb,74.000,Public internet egress.
5,object_storage,2024-01-01,tb_month_avg,22.000,Average stored TB per month.
6,object_storage,2026-01-01,tb_month_avg,20.500,Storage price refresh.


In [4]:
fig = px.line(
    df_prices, x="effective_from", y="list_price_usd",
    color="product", markers=True,
    facet_col="product", facet_col_wrap=3,
    title="Price history by product",
    labels={"effective_from": "Date", "list_price_usd": "List price (USD)", "product": "Product"}
)
fig.update_yaxes(matches=None)
fig.write_image("block1_price_dynamics.png")
fig.show()

**Findings:**

The table has 7 rows across 5 products. Three price changes over the period:
- `compute_standard`: cut from 0.046 → 0.043 effective July 2025
- `gpu_a100` → `gpu_accelerated`: rename + cut 2.65 → 2.55 effective October 2025
- `object_storage`: cut from 22.00 → 20.50 effective January 2026
- `network_egress`: price stable (74.00 per egress_tb)

GPU is a single product: `gpu_a100` before October 2025, `gpu_accelerated` after.

> **📋 Calculation note:** Net revenue requires a range join with price_list via `LEAD()`: `month >= effective_from AND (next_date IS NULL OR month < next_date)`. Combine GPU into a single cluster for historical analysis.


### 1.2 Customer base — segments and statuses


In [5]:
df_cust = con.sql("""
    SELECT segment, customer_status, COUNT(*) AS customers,
           ROUND(AVG(payment_terms_days), 0) AS avg_payment_terms_days
    FROM billing_customers
    GROUP BY segment, customer_status
    ORDER BY segment, customer_status
""").df()
df_cust

,segment,customer_status,customers,avg_payment_terms_days
0,Commercial,active,25,35.0
1,Commercial,churned,3,40.0
2,Commercial,paused,2,45.0
3,Enterprise,active,12,40.0
4,Enterprise,churned,1,30.0
5,SMB,active,26,38.0
6,SMB,churned,3,40.0
7,SMB,paused,2,30.0
8,Strategic,active,16,34.0


In [6]:
fig = px.bar(
    df_cust, x="segment", y="customers", color="customer_status",
    barmode="stack",
    title="Customers by segment and status",
    labels={"segment": "Segment", "customers": "Count", "customer_status": "Status"}
)
fig.write_image("block1_customers_by_segment.png")
fig.show()

**Findings:**

90 customers in total. Churned and paused break down by segment per the table above. The Strategic segment has no churn. Payment terms differ by segment — important for the cash collections model.

> **📋 Calculation note:** Exclude churned and paused customers from the baseline. Separately check which churned customers still have usage in May 2026 — remove their revenue from the forecast base.


### 1.3 Link between billing_customers and monthly_usage


In [7]:
con.sql("""
    SELECT
        COUNT(DISTINCT mu.customer_id) AS customers_in_usage,
        COUNT(DISTINCT bc.customer_id) AS customers_in_billing,
        COUNT(DISTINCT CASE WHEN bc.customer_id IS NULL THEN mu.customer_id END) AS in_usage_not_billing,
        COUNT(DISTINCT CASE WHEN mu.customer_id IS NULL THEN bc.customer_id END) AS in_billing_not_usage
    FROM monthly_usage mu
    FULL OUTER JOIN billing_customers bc ON mu.customer_id = bc.customer_id
""").df()

,customers_in_usage,customers_in_billing,in_usage_not_billing,in_billing_not_usage
0,90,90,0,0


**Findings:**

Integrity check of the link: every customer in monthly_usage is present in billing_customers and vice versa. No orphan records.

> **📋 Calculation note:** The tables can be joined on customer_id without loss.


### 1.4 Paused and churned customer statuses — last consumption date


In [8]:
con.sql("""
    SELECT bc.customer_status, bc.customer_id, bc.customer_name, bc.segment,
           MAX(mu.month) AS last_usage_date
    FROM billing_customers bc
    LEFT JOIN monthly_usage mu ON bc.customer_id = mu.customer_id
    WHERE bc.customer_status IN ('paused', 'churned')
    GROUP BY bc.customer_status, bc.customer_id, bc.customer_name, bc.segment
    ORDER BY bc.customer_status, last_usage_date DESC
""").df()

,customer_status,customer_id,customer_name,segment,last_usage_date
0,churned,CUST-0011,MedNova,Commercial,2026-05-01
1,churned,CUST-0016,PeakMetrics,Commercial,2026-05-01
2,churned,CUST-0037,Aurora Systems Ltd.,SMB,2026-05-01
3,churned,CUST-0012,BrightApps,SMB,2026-05-01
4,churned,CUST-0044,Northstar Gaming Inc.,Commercial,2026-02-01
5,churned,CUST-0080,DeltaStream Technologies,Enterprise,2026-01-01
6,churned,CUST-0014,Atlas Retail,SMB,2025-10-01
7,paused,CUST-0046,CloudNest Inc.,Commercial,2026-01-01
8,paused,CUST-0004,Northstar Gaming,SMB,2026-01-01
9,paused,CUST-0056,PeakMetrics Inc.,Commercial,2026-01-01


**Findings:**

Several key patterns are visible:
- **Paused**: last consumption in January 2026 — these customers have long stopped using the product.
- **Churned**: some customers have their last consumption in May 2026 — still active in the last historical month, but already flagged "churned".

> **📋 Calculation note:** (1) Exclude all paused and churned from the baseline. (2) Special attention — churned customers with last_usage = May 2026: they contribute to May but not to the forecast period. The baseline must account for this.


### 1.5 Historical usage data coverage


In [9]:
con.sql("""
    SELECT
        MIN(month) AS first_month,
        MAX(month) AS last_month,
        COUNT(DISTINCT month) AS total_months,
        COUNT(DISTINCT customer_id) AS distinct_customers,
        COUNT(DISTINCT product) AS distinct_products,
        COUNT(*) AS total_rows
    FROM monthly_usage
""").df()

,first_month,last_month,total_months,distinct_customers,distinct_products,total_rows
0,2024-01-01,2026-05-01,29,90,5,8511


**Findings:**

29 months of history (January 2024 — May 2026), 90 customers, 5 products, 8,511 rows. No month gaps. The forecast period starts in June 2026.

> **📋 Calculation note:** Build the baseline on April–May 2026.


### 1.6 Active customers by month


In [10]:
df_active = con.sql("""
    SELECT month, COUNT(DISTINCT customer_id) AS active_customers
    FROM monthly_usage
    GROUP BY month
    ORDER BY month
""").df()

fig = px.line(
    df_active, x="month", y="active_customers", markers=True,
    title="Active customers per month",
    labels={"month": "Month", "active_customers": "Active customers"}
)
fig.write_image("block1_active_customers.png")
fig.show()

**Findings:**

The active customer base grew steadily through mid-2025, then stabilized with minor fluctuations. Dips in individual months may reflect either churned customers or a temporary absence of usage.

> **📋 Calculation note:** Include customers with usage in April and/or May 2026 in the baseline.


### 1.6b Active customers by product


In [11]:
df_active_prod = con.sql("""
    SELECT month,
           CASE product
               WHEN 'gpu_a100'        THEN 'gpu'
               WHEN 'gpu_accelerated' THEN 'gpu'
               ELSE product
           END AS product_group,
           COUNT(DISTINCT customer_id) AS active_customers
    FROM monthly_usage
    GROUP BY month, product_group
    ORDER BY month, product_group
""").df()

fig = px.line(
    df_active_prod, x="month", y="active_customers", color="product_group",
    title="Active customers per product per month",
    labels={"month": "Month", "active_customers": "Active customers", "product_group": "Product"}
)
fig.write_image("block1_active_customers_per_product.png")
fig.show()

**Findings:**

The distribution of active customers by product shows which products are used broadly and which by a limited set of customers. GPU is used by far fewer customers than compute or storage.

> **📋 Calculation note:** GPU is a niche product with few customers but high potential revenue per customer.


### 1.6c Number of products per customer


In [12]:
df_prod_cnt = con.sql("""
    WITH cte AS (
        SELECT month, customer_id,
               COUNT(DISTINCT CASE WHEN product LIKE '%gpu%' THEN 'gpu' ELSE product END) AS n_products
        FROM monthly_usage
        GROUP BY month, customer_id
    )
    SELECT month,
           CASE n_products
               WHEN 1 THEN '1 product'
               WHEN 2 THEN '2 products'
               WHEN 3 THEN '3 products'
               WHEN 4 THEN '4 products'
           END AS product_count_group,
           COUNT(DISTINCT customer_id) AS customers
    FROM cte
    GROUP BY month, product_count_group
    ORDER BY month, product_count_group
""").df()

fig = px.line(
    df_prod_cnt, x="month", y="customers", color="product_count_group",
    title="Customers by number of products used per month",
    labels={"month": "Month", "customers": "Customers", "product_count_group": "Product count"}
)
fig.write_image("block1_products_per_customer.png")
fig.show()

**Findings:**

Most customers consume all 4 product groups at once. A small share uses only 3 products. Customers with 1–2 products are rare.

> **📋 Calculation note:** Cross-product dependency means that churning a single customer loses revenue across several products at once.


### 1.7 Historical revenue — dynamics by product


In [13]:
df_rev = con.sql("""
    WITH price_eff AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS effective_to,
               list_price_usd
        FROM price_list
    ),
    usage_qty AS (
        SELECT month, customer_id,
               CASE product
                   WHEN 'gpu_a100'        THEN 'gpu'
                   WHEN 'gpu_accelerated' THEN 'gpu'
                   ELSE product
               END AS product_group,
               CASE product
                   WHEN 'compute_standard' THEN compute_hours
                   WHEN 'gpu_a100'         THEN gpu_hours
                   WHEN 'gpu_accelerated'  THEN gpu_hours
                   WHEN 'object_storage'   THEN storage_tb_avg
                   WHEN 'network_egress'   THEN egress_tb
               END AS qty,
               product AS orig_product
        FROM monthly_usage
    )
    SELECT
        u.month,
        u.product_group,
        ROUND(SUM(u.qty * p.list_price_usd
              * (1 - COALESCE(c.discount_pct, 0) / 100)) / 1000, 1) AS revenue_k
    FROM usage_qty u
    JOIN price_eff p
        ON u.orig_product = p.product
        AND u.month >= p.effective_from
        AND (p.effective_to IS NULL OR u.month < p.effective_to)
    LEFT JOIN contracts c
        ON u.customer_id = c.customer_id
        AND u.month >= DATE_TRUNC('month', c.contract_start)
        AND u.month <= DATE_TRUNC('month', c.contract_end)
    GROUP BY u.month, u.product_group
    ORDER BY u.month, u.product_group
""").df()

fig = px.area(
    df_rev, x="month", y="revenue_k", color="product_group",
    title="Historical consumption revenue by product ($K/month)",
    labels={"month": "Month", "revenue_k": "Revenue ($K)", "product_group": "Product"}
)
fig.write_image("block1_revenue_by_product.png")
fig.show()

In [14]:
# Revenue structure for May 2026 — the last historical month
con.sql("""
    WITH price_eff AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS effective_to,
               list_price_usd
        FROM price_list
    ),
    usage_qty AS (
        SELECT month, customer_id,
               CASE product WHEN 'gpu_a100' THEN 'gpu' WHEN 'gpu_accelerated' THEN 'gpu' ELSE product END AS product_group,
               CASE product
                   WHEN 'compute_standard' THEN compute_hours
                   WHEN 'gpu_a100'         THEN gpu_hours
                   WHEN 'gpu_accelerated'  THEN gpu_hours
                   WHEN 'object_storage'   THEN storage_tb_avg
                   WHEN 'network_egress'   THEN egress_tb
               END AS qty,
               product AS orig_product
        FROM monthly_usage
        WHERE month = '2026-05-01'
    ),
    rev AS (
        SELECT u.product_group,
               SUM(u.qty * p.list_price_usd * (1 - COALESCE(c.discount_pct, 0) / 100)) AS revenue
        FROM usage_qty u
        JOIN price_eff p ON u.orig_product = p.product
            AND u.month >= p.effective_from AND (p.effective_to IS NULL OR u.month < p.effective_to)
        LEFT JOIN contracts c
            ON u.customer_id = c.customer_id
            AND u.month >= DATE_TRUNC('month', c.contract_start)
            AND u.month <= DATE_TRUNC('month', c.contract_end)
        GROUP BY u.product_group
    )
    SELECT product_group,
           ROUND(revenue / 1000, 1) AS revenue_k,
           ROUND(100.0 * revenue / SUM(revenue) OVER (), 1) AS pct_of_total
    FROM rev
    ORDER BY revenue DESC
""").df()

,product_group,revenue_k,pct_of_total
0,network_egress,867.5,59.2
1,object_storage,304.8,20.8
2,gpu,206.7,14.1
3,compute_standard,87.4,6.0


**Findings:**

Revenue grew steadily throughout the period. The product structure and exact shares are in the table above (May 2026). The price cuts (compute in July 2025, gpu in October 2025, storage in January 2026) slowed growth in dollar terms despite rising volumes.

> **📋 Calculation note:** GPU is more volatile than the other products — forecast it separately. The product revenue mix (from the table above) is a reference for allocating pipeline effects.


> 📎 A detailed breakdown of usage for each product (GPU, Compute, Object Storage, Network Egress) by customer is moved to **Appendix A** at the end of the notebook — to keep the main narrative concise.


### 1.8 Contracts — statuses, billing, discounts, and link to usage


In [15]:
con.sql("""
    SELECT c.contract_status, c.billing_frequency, c.auto_renewal,
           DATE_TRUNC('month', c.contract_end) AS contract_end_month,
           MAX(m.month) AS last_usage_date,
           COUNT(DISTINCT c.customer_id) AS customers
    FROM contracts c
    INNER JOIN monthly_usage m ON c.customer_id = m.customer_id
    GROUP BY c.contract_status, c.billing_frequency, c.auto_renewal,
             DATE_TRUNC('month', c.contract_end)
    ORDER BY c.contract_status, c.billing_frequency, c.auto_renewal
""").df()

,contract_status,billing_frequency,auto_renewal,contract_end_month,last_usage_date,customers
0,active,annual_prepay,False,2025-06-01,2026-05-01,2
1,active,annual_prepay,False,2027-06-01,2026-05-01,1
2,active,annual_prepay,False,2026-11-01,2026-01-01,1
3,active,annual_prepay,False,2027-03-01,2026-05-01,1
4,active,annual_prepay,False,2026-01-01,2026-05-01,1
5,active,annual_prepay,False,2027-02-01,2026-05-01,1
6,active,annual_prepay,True,2027-08-01,2026-05-01,1
7,active,annual_prepay,True,2027-02-01,2026-05-01,1
8,active,annual_prepay,True,2027-11-01,2026-05-01,1
9,active,annual_prepay,True,2025-12-01,2026-05-01,1


**Findings:**

Most contracted customers are active through May 2026. There are contracts with `auto_renewal = False` and `contract_end` before June 2026 — for some of them last consumption is also in May, i.e. the customer is active but the contract is de jure not renewed. There is 1 contract with status `expired`. The `signed_not_started` contract has not started yet.

> **📋 Calculation note:** Customers with `auto_renewal = False` and `contract_end` in June–August 2026 — after expiry their discount_pct = 0. This must be applied in the forecast.


### 1.9 Contracts — overview by billing type


In [16]:
con.sql("""
    SELECT billing_frequency, contract_status, auto_renewal,
           COUNT(*) AS contracts,
           ROUND(AVG(discount_pct), 1) AS avg_discount_pct,
           ROUND(AVG(payment_terms_days), 0) AS avg_payment_days,
           ROUND(SUM(committed_spend_usd) / 1e6, 2) AS total_committed_m
    FROM contracts
    GROUP BY billing_frequency, contract_status, auto_renewal
    ORDER BY billing_frequency, contract_status, auto_renewal
""").df()

,billing_frequency,contract_status,auto_renewal,contracts,avg_discount_pct,avg_payment_days,total_committed_m
0,annual_prepay,active,False,7,9.4,34.0,1.55
1,annual_prepay,active,True,8,12.1,32.0,10.13
2,annual_prepay,signed_not_started,True,1,20.0,45.0,1.80
3,monthly,active,False,8,11.6,30.0,13.02
4,monthly,active,True,5,12.6,36.0,3.77
5,monthly,expired,False,1,18.0,30.0,0.57
6,quarterly,active,False,11,7.9,43.0,6.43
7,quarterly,active,True,5,9.0,30.0,5.92


**Findings:**

Three main billing types: annual_prepay, quarterly, monthly. `signed_not_started` (annual_prepay) — the contract is awaiting activation, likely corresponding to opportunity OPP-1039 in CRM. Average discounts are 8–12% depending on type.

> **📋 Calculation note:** `billing_frequency` determines cash timing: annual_prepay — a single payment at the start of the term; quarterly — 4 payments a year; monthly — a lag of ~2 months. Segment customers when building the cash model.


### 1.10 Contracts expiring in the forecast period (June–August 2026)


In [17]:
con.sql("""
    SELECT c.contract_id, c.customer_id, bc.customer_name, bc.segment,
           c.billing_frequency, c.discount_pct, c.auto_renewal, c.contract_end
    FROM contracts c
    JOIN billing_customers bc ON c.customer_id = bc.customer_id
    WHERE DATE_TRUNC('month', c.contract_end) BETWEEN '2026-06-01' AND '2026-08-31'
    ORDER BY c.contract_end, c.auto_renewal
""").df()

,contract_id,customer_id,customer_name,segment,billing_frequency,discount_pct,auto_renewal,contract_end
0,CON-1011,CUST-0019,OmniPay,SMB,monthly,12,True,2026-08-31


**Findings:**

Contracts with `auto_renewal = False` expiring in June–August 2026 move the customer to list price (discount_pct = 0) after expiry. A small positive effect on revenue after the expiry date.

> **📋 Calculation note:** For each such contract: in the expiry month apply the discount only pro rata, and from the following month set discount = 0.


### 1.11 Customers without an active contract


In [18]:
con.sql("""
    SELECT bc.segment, bc.customer_status, COUNT(*) AS customers
    FROM billing_customers bc
    LEFT JOIN contracts c
        ON bc.customer_id = c.customer_id
        AND c.contract_status = 'active'
    WHERE c.contract_id IS NULL
    GROUP BY bc.segment, bc.customer_status
    ORDER BY bc.segment, bc.customer_status
""").df()

,segment,customer_status,customers
0,Commercial,active,17
1,Commercial,churned,3
2,Commercial,paused,2
3,Enterprise,churned,1
4,SMB,active,19
5,SMB,churned,3
6,SMB,paused,1


**Findings:**

A significant part of the base (mostly SMB and Commercial) has no active contract — they consume at list price with no discount.

> **📋 Calculation note:** `LEFT JOIN contracts WHERE contract_status = 'active'` + `COALESCE(discount_pct, 0)` for all customers without a contract.


### 1.12 Invoices — statuses and amounts


In [19]:
con.sql("""
    SELECT payment_status, invoice_type,
           COUNT(*) AS invoices,
           ROUND(SUM(amount_usd) / 1e6, 3) AS total_amount_m_usd
    FROM invoices
    GROUP BY payment_status, invoice_type
    ORDER BY payment_status, invoice_type
""").df()

,payment_status,invoice_type,invoices,total_amount_m_usd
0,open,usage,129,1.612
1,paid,annual_commit_prepay,14,10.049
2,paid,billing_correction,1,-0.084
3,paid,service_credit,1,-0.215
4,paid,sla_credit,1,-0.036
5,paid,usage,2015,26.825
6,void_pending,test_invoice,1,73.843


**Findings:**

From the table above — several groups:
1. **void_pending / test_invoice** — $73.8M. A test record, exclude everywhere.
2. **open / usage** — 129 invoices for $1.6M. Unpaid receivables.
3. **paid / usage** — 2,015 invoices for $26.8M. The bulk of historical revenue — this is where the payment lag distribution comes from.
4. **paid / annual_commit_prepay** — $10M. Historical prepay receipts.
5. **paid / billing_correction, service_credit, sla_credit** — adjustments, immaterial in total (< $0.4M).

> **📋 Calculation note:** Exclude `void_pending` everywhere. Study the billing lag on `paid/usage`. Analyze the $1.6M of open receivables in the next section.


### 1.13 Open receivables — aging analysis


In [20]:
df_aging = con.sql("""
    SELECT
        CASE
            WHEN DATEDIFF('day', due_date, DATE '2026-06-06') BETWEEN 1  AND 30  THEN '01. 1–30 days'
            WHEN DATEDIFF('day', due_date, DATE '2026-06-06') BETWEEN 31 AND 90  THEN '02. 31–90 days'
            WHEN DATEDIFF('day', due_date, DATE '2026-06-06') BETWEEN 91 AND 180 THEN '03. 91–180 days'
            WHEN DATEDIFF('day', due_date, DATE '2026-06-06') > 180              THEN '04. >180 days'
            ELSE '00. Within terms'
        END AS aging_bucket,
        COUNT(*) AS invoices,
        ROUND(SUM(amount_usd) / 1000, 1) AS amount_k_usd
    FROM open_ar_asof
    WHERE due_date + INTERVAL 7 DAY < '2026-06-01'   -- overdue part of AR as of the cutoff (same set as in recovery)
    GROUP BY aging_bucket
    ORDER BY aging_bucket
""").df()

fig = px.bar(
    df_aging, x="aging_bucket", y="amount_k_usd", text="invoices",
    title="Open invoice aging as of June 2026 ($K)",
    labels={"aging_bucket": "Overdue bucket", "amount_k_usd": "Amount ($K)", "invoices": "# invoices"}
)
fig.update_traces(textposition="outside")
fig.write_image("block1_open_invoice_aging.png")
fig.show()


**Findings:**

Overdue receivables as of the cutoff date (as-of reconstruction, the same set used in the cash model): **126 invoices for ~$1,442K**, all overdue. ~70% ($999K) is debt older than 180 days.

> **📋 Calculation note:** (1) >180 days — effectively bad debt (recovery 5%). (2) Fresher buckets — recovery 80%/50%/20% (1–30 / 31–90 / 91–180 days). (3) Rates are scenario assumptions, to be validated with AR.


### 1.14 CRM accounts — quality of the billing link


In [21]:
con.sql("""
    SELECT match_confidence,
           COUNT(*) AS crm_accounts,
           COUNT(matched_customer_id) AS linked_to_billing,
           COUNT(*) - COUNT(matched_customer_id) AS unlinked
    FROM crm_accounts
    GROUP BY match_confidence
    ORDER BY match_confidence
""").df()

,match_confidence,crm_accounts,linked_to_billing,unlinked
0,high,31,31,0
1,low,17,17,0
2,medium,24,24,0
3,new_logo,4,0,4
4,unmatched,3,0,3


**Findings:**

Accounts with `high`, `medium`, `low` confidence are linked to billing_customers. `new_logo` and `unmatched` are unlinked (potential new customers).

> **📋 Calculation note:** For expansion deals, join via `matched_customer_id`. For `new_logo` — revenue from scratch. With `low` confidence, use the match cautiously.


### 1.15 Pipeline — sales funnel overview


In [22]:
df_pip = con.sql("""
    SELECT opportunity_type, stage,
           COUNT(*) AS opps,
           ROUND(AVG(probability), 2) AS avg_prob,
           ROUND(SUM(expected_monthly_consumption_usd) / 1000, 1) AS total_monthly_k,
           ROUND(SUM(expected_monthly_consumption_usd * probability) / 1000, 1) AS weighted_monthly_k
    FROM crm_opportunities
    GROUP BY opportunity_type, stage
    ORDER BY weighted_monthly_k DESC
""").df()
df_pip

,opportunity_type,stage,opps,avg_prob,total_monthly_k,weighted_monthly_k
0,Expansion,Commit,8,0.78,1765.3,1555.7
1,New Logo,Negotiation,3,0.60,240.8,160.2
2,Expansion,Closed Won,1,1.00,150.0,150.0
3,Renewal,Discovery,3,0.65,301.5,96.8
4,Renewal,Solution Fit,4,0.51,140.1,76.1
5,Expansion,Proposal,2,0.55,110.5,68.4
6,New Logo,Proposal,5,0.49,109.5,56.3
7,New Logo,Discovery,4,0.40,143.0,45.8
8,Expansion,Solution Fit,3,0.45,108.0,44.9
9,New Logo,Commit,1,0.55,36.3,19.9


In [23]:
df_pip_chart = con.sql("""
    SELECT stage, forecast_category,
           ROUND(SUM(expected_monthly_consumption_usd) / 1000, 1) AS monthly_k
    FROM crm_opportunities
    GROUP BY stage, forecast_category
    ORDER BY monthly_k DESC
""").df()

fig = px.bar(
    df_pip_chart, x="stage", y="monthly_k", color="forecast_category",
    title="CRM pipeline: total monthly consumption by stage and forecast category ($K)",
    labels={"stage": "Stage", "monthly_k": "Monthly revenue ($K)", "forecast_category": "Forecast category"}
)
fig.write_image("block1_pipeline_by_stage.png")
fig.show()

**Findings:**

The funnel: Expansion, New Logo, Renewal. The largest weighted potential is Expansion at the Commit stage. Total pipeline — per the table above.

> **📋 Calculation note:** For the forecast — only deals with `expected_start_month` in June–August 2026. Base: weighted by probability. Upside: Commit/Closed Won at full value.


### 1.16 Pipeline — deals starting in June–August 2026


In [24]:
con.sql("""
    SELECT o.opportunity_id, o.opportunity_name, o.opportunity_type,
           o.stage, o.probability, o.expected_close_date,
           o.expected_start_month, o.expected_monthly_consumption_usd,
           o.payment_structure, o.owner_forecast_notes
    FROM crm_opportunities o
    WHERE o.expected_start_month BETWEEN '2026-06-01' AND '2026-08-01'
    ORDER BY o.expected_start_month, o.probability DESC
""").df()

,opportunity_id,opportunity_name,opportunity_type,stage,probability,expected_close_date,expected_start_month,expected_monthly_consumption_usd,payment_structure,owner_forecast_notes
0,OPP-1039,Aurora Systems signed annual prepay not in bil...,Expansion,Closed Won,1.00,2026-05-28,2026-06-01,150000,annual_prepay,Contract signed but first invoice is not prese...
1,OPP-1035,Vertex Labs strategic gpu_reserved_capacity,Expansion,Commit,0.90,2026-06-24,2026-07-01,410000,annual_prepay,Part of same board-approved AI platform package.
2,OPP-1036,Vertex Labs strategic compute_savings_plan,Expansion,Commit,0.90,2026-06-24,2026-07-01,230000,annual_prepay,May be bundled into master commit.
3,OPP-1037,Vertex Labs strategic cloud_commit_plan,Expansion,Commit,0.90,2026-06-24,2026-07-01,690000,annual_prepay,Umbrella saving plan; avoid double counting ch...
4,OPP-1017,Tesseract AI Ltd. expansion object_storage_commit,Renewal,Negotiation,0.55,2026-06-16,2026-07-01,21703,commit_drawdown,Sales expects fast ramp.
5,OPP-1031,Lambda Robotics expansion compute_savings_plan,New Logo,Discovery,0.40,2026-06-26,2026-07-01,24464,monthly_arrears,Legal review in progress.
6,OPP-1005,Atlas Retail expansion gpu_reserved_capacity,Expansion,Solution Fit,0.25,2026-07-15,2026-07-01,24492,annual_prepay,Sales expects fast ramp.
7,OPP-1030,BrightApps Cloud expansion object_storage_commit,Renewal,Discovery,0.85,2026-08-17,2026-08-01,23065,monthly_arrears,Procurement timing uncertain.
8,OPP-1021,Finwise Inc. expansion cloud_commit_plan,Renewal,Solution Fit,0.70,2026-07-24,2026-08-01,56808,quarterly_prepay,Legal review in progress.
9,OPP-1038,Lambda Robotics new logo GPU cloud launch,New Logo,Negotiation,0.70,2026-07-18,2026-08-01,185000,quarterly_prepay,No billing customer exists yet; sales says lau...


**Findings:**

12 deals starting in June–August 2026. From the table above, the standouts:

- **OPP-1039** (Closed Won, 150K/mo, `annual_prepay`, June start) — the only closed deal. Corresponds to the `signed_not_started` contract in contracts.
- **OPP-1035, OPP-1036, OPP-1037** — all three tied to a single account, Vertex Labs, with the same close date (June 24) and the same start (July). ~1,330K/mo combined. The `owner_forecast_notes` field holds information about the nature of these deals — it must be read before including them in the model.
- **OPP-1038** (Lambda Robotics, New Logo, 185K/mo, 70% probability, August start) — a significant new deal.

> **📋 Calculation note:** (1) OPP-1039: 150K/mo in all scenarios + 1.8M cash prepay in June. (2) OPP-1035/1036/1037: be sure to read `owner_forecast_notes` in the output above and validate with Sales — three deals on one account with the same close date under the new sales policy (May 2026) is a classic sign of double counting. (3) The rest — `probability × expected_monthly_consumption_usd`.


### 1.17 CRM Opportunity Products — product mix of deals


In [25]:
# Inspect the table structure
con.sql("""
    SELECT *
    FROM crm_opportunity_products
    LIMIT 5
""").df()

,opportunity_id,crm_sku,product_family,estimated_monthly_usage,estimated_monthly_revenue_usd,ramp_month,confidence
0,OPP-1001,gpu_reserved_capacity,gpu_accelerated,7615.02,3630,3,medium
1,OPP-1002,compute_savings_plan,compute_standard,21704.08,7316,3,medium
2,OPP-1003,compute_savings_plan,compute_standard,9190.92,2213,2,medium
3,OPP-1004,cloud_commit_plan,multi_product_commit,6919.98,6747,1,high
4,OPP-1005,gpu_reserved_capacity,gpu_accelerated,20384.77,5820,1,medium


In [26]:
# Pipeline product breakdown: what type of consumption the deals assume
con.sql("""
    SELECT op.product_family, op.crm_sku, op.confidence,
           COUNT(DISTINCT op.opportunity_id)                            AS opps,
           ROUND(SUM(op.estimated_monthly_revenue_usd) / 1000, 1)      AS total_monthly_k,
           ROUND(AVG(op.estimated_monthly_revenue_usd) / 1000, 1)      AS avg_monthly_k,
           ROUND(AVG(op.ramp_month), 1)                                AS avg_ramp_month
    FROM crm_opportunity_products op
    JOIN crm_opportunities o ON op.opportunity_id = o.opportunity_id
    GROUP BY op.product_family, op.crm_sku, op.confidence
    ORDER BY total_monthly_k DESC
""").df()

,product_family,crm_sku,confidence,opps,total_monthly_k,avg_monthly_k,avg_ramp_month
0,multi_product_commit,cloud_commit_plan,high,4,862.1,215.5,1.0
1,gpu_accelerated,gpu_reserved_capacity,high,2,419.3,209.7,2.0
2,gpu_accelerated,gpu_reserved_capacity,medium,6,330.1,55.0,2.2
3,compute_standard,compute_savings_plan,high,2,241.9,121.0,1.0
4,object_storage,object_storage_commit,high,4,212.0,53.0,1.5
5,multi_product_commit,cloud_commit_plan,medium,5,161.0,32.2,1.0
6,compute_standard,compute_savings_plan,medium,5,70.9,14.2,2.2
7,multi_product_commit,cloud_commit_plan,low,3,63.6,21.2,1.3
8,network_egress,network_commit,high,2,38.0,19.0,2.0
9,gpu_accelerated,gpu_reserved_capacity,low,2,35.7,17.9,2.0


In [27]:
# Check: does the product sum match the opportunity level?
con.sql("""
    SELECT o.opportunity_id, o.opportunity_name,
           ROUND(o.expected_monthly_consumption_usd, 0)        AS opp_total,
           ROUND(SUM(op.estimated_monthly_revenue_usd), 0)     AS products_sum,
           ROUND(
               o.expected_monthly_consumption_usd
               - SUM(op.estimated_monthly_revenue_usd), 0
           )                                                    AS delta
    FROM crm_opportunities o
    LEFT JOIN crm_opportunity_products op ON o.opportunity_id = op.opportunity_id
    WHERE o.expected_start_month BETWEEN '2026-06-01' AND '2026-08-01'
    GROUP BY o.opportunity_id, o.opportunity_name,
             o.expected_monthly_consumption_usd
    ORDER BY ABS(delta) DESC
""").df()

,opportunity_id,opportunity_name,opp_total,products_sum,delta
0,OPP-1021,Finwise Inc. expansion cloud_commit_plan,56808,20195.0,36613.0
1,OPP-1005,Atlas Retail expansion gpu_reserved_capacity,24492,5820.0,18672.0
2,OPP-1030,BrightApps Cloud expansion object_storage_commit,23065,9934.0,13131.0
3,OPP-1009,HelioBank Ltd. expansion gpu_reserved_capacity,18694,9347.0,9347.0
4,OPP-1017,Tesseract AI Ltd. expansion object_storage_commit,21703,15487.0,6216.0
5,OPP-1011,Orbit Media Ltd. expansion object_storage_commit,9736,5296.0,4440.0
6,OPP-1031,Lambda Robotics expansion compute_savings_plan,24464,21784.0,2680.0
7,OPP-1035,Vertex Labs strategic gpu_reserved_capacity,410000,410000.0,0.0
8,OPP-1036,Vertex Labs strategic compute_savings_plan,230000,230000.0,0.0
9,OPP-1038,Lambda Robotics new logo GPU cloud launch,185000,185000.0,0.0


**Findings:**

The `crm_opportunity_products` table contains a product breakdown of each deal. The reconciliation check (delta = opp_total − products_sum) shows:
- If delta = 0 everywhere: product rows fully explain the opportunity total → usable for product-level forecasting.
- If delta ≠ 0: opportunity total and products do not reconcile → the product breakdown is unreliable, use only opportunity-level figures.

> **📋 Calculation note:** Where reconciliation differences exist — bring the pipeline into the revenue calculation at the opportunity level (the `expected_monthly_consumption_usd` field), not via products. Use the product breakdown only to allocate across products in the revenue mix, not as the primary source.


### 1.18 Billing lag — empirical distribution over paid invoices


In [28]:
# Inspect invoice fields with dates
con.sql("""
    SELECT *
    FROM clean_invoices
    WHERE payment_status = 'paid' AND invoice_type = 'usage'
    LIMIT 5
""").df()

,invoice_id,customer_id,invoice_date,service_month,contract_id,invoice_type,amount_usd,due_date,paid_date,payment_status
0,INV-70006,CUST-0001,2024-08-03,2024-07-01,NaN,usage,21325.45,2024-09-17,2024-09-24,paid
1,INV-70031,CUST-0002,2025-06-03,2025-05-01,CON-1001,usage,3559.60,2025-08-02,2025-08-02,paid
2,INV-70036,CUST-0002,2025-11-03,2025-10-01,NaN,usage,3600.34,2026-01-02,2026-01-09,paid
3,INV-70037,CUST-0002,2025-12-02,2025-11-01,NaN,usage,3653.14,2026-01-31,2026-01-26,paid
4,INV-70052,CUST-0003,2024-10-06,2024-09-01,NaN,usage,12496.26,2024-11-20,2025-01-04,paid


In [29]:
# Lag between invoice date and payment date (in months)
df_lag = con.sql("""
    SELECT
        DATEDIFF('month',
            DATE_TRUNC('month', invoice_date),
            DATE_TRUNC('month', paid_date)
        ) AS lag_months,
        COUNT(*) AS invoices,
        ROUND(SUM(amount_usd) / 1000, 1) AS amount_k
    FROM clean_invoices
    WHERE payment_status = 'paid'
      AND invoice_type = 'usage'
      AND paid_date IS NOT NULL
      AND paid_date <= DATE '2026-06-06'
      AND invoice_date IS NOT NULL
    GROUP BY lag_months
    ORDER BY lag_months
""").df()
df_lag

,lag_months,invoices,amount_k
0,-1,1,12.4
1,0,346,4234.8
2,1,968,13690.1
3,2,485,5675.9
4,3,70,789.3


In [30]:
# Share by lag as a percentage of amount
con.sql("""
    WITH base AS (
        SELECT
            DATEDIFF('month',
                DATE_TRUNC('month', invoice_date),
                DATE_TRUNC('month', paid_date)
            ) AS lag_months,
            amount_usd
        FROM clean_invoices
        WHERE payment_status = 'paid'
          AND invoice_type = 'usage'
          AND paid_date IS NOT NULL
          AND paid_date <= DATE '2026-06-06'
          AND invoice_date IS NOT NULL
    )
    SELECT lag_months,
           COUNT(*) AS invoices,
           ROUND(SUM(amount_usd) / 1000, 1) AS amount_k,
           ROUND(100.0 * SUM(amount_usd) / SUM(SUM(amount_usd)) OVER (), 1) AS pct_of_total
    FROM base
    GROUP BY lag_months
    ORDER BY lag_months
""").df()

,lag_months,invoices,amount_k,pct_of_total
0,-1,1,12.4,0.1
1,0,346,4234.8,17.4
2,1,968,13690.1,56.1
3,2,485,5675.9,23.3
4,3,70,789.3,3.2


In [31]:
fig = px.bar(
    df_lag, x="lag_months", y="amount_k", text="invoices",
    title="Payment lag distribution: paid usage invoices (months from invoice to payment)",
    labels={"lag_months": "Lag (months)", "amount_k": "Amount ($K)", "invoices": "# invoices"}
)
fig.update_traces(textposition="outside")
fig.write_image("block1_billing_lag_distribution.png")
fig.show()

**Findings:**

Empirical distribution of the lag between invoice date and payment date over historical paid invoices (table above). The distribution shows:
- Most payments arrive 1–2 months after the invoice is issued.
- The share attributable to each lag bucket is shown in the `pct_of_total` column.

> **📋 Calculation note:** Use the empirical weights from `pct_of_total` for the cash collections model. Formula: *cash(month T) = rev(T−0) × w0 + rev(T−1) × w1 + rev(T−2) × w2 + ...*, where w0, w1, w2 are the shares from the table at lag 0, 1, 2 months respectively. Segment by `billing_frequency` if needed — prepay customers (annual_commit_prepay) have lag = 0 and are handled separately.


---
## Block 1 summary — key flags for the calculations

| # | Flag | Action in the calculations |
|---|---|---|
| 1 | Three price changes over the period | Range join with price_list via `LEAD()` |
| 2 | gpu_a100 → gpu_accelerated (October 2025) | Single GPU product, continuous trend |
| 3 | GPU capacity constraint (March 2026) | GPU baseline — April–May, exclude March |
| 4 | Billing system change (April 2026) | April invoices — treat with caution |
| 5 | New sales policy (May 2026) | Check pipeline for double counting |
| 6 | Churned with usage in May 2026 | Contributes to May, not to the forecast |
| 7 | Paused customers (no usage since Jan 2026) | Exclude from baseline |
| 8 | Customers without an active contract | `discount_pct = 0` |
| 9 | Contracts without `auto_renewal` expiring in June–August | Discount = 0 after expiry |
| 10 | Test invoice 73.8M (`void_pending`) | Exclude everywhere |
| 11 | Overdue receivables 1.6M (all > 30 days) | >180 days — bad debt; the rest recovery 10–20% |
| 12 | OPP-1035/1036/1037 (Vertex Labs) | Read `owner_forecast_notes`, validate with Sales |
| 13 | OPP-1039 (Closed Won, Aurora Systems) | 150K/mo + 1.8M prepay cash in June |
| 14 | crm_opportunity_products reconciliation | If delta ≠ 0 — pipeline by opportunity total, not by products |
| 15 | Billing lag (empirical) | Weights from the lag distribution → cash collections formula |


---
## Block 2 — Historical revenue

We compute actual consumption revenue over the full history using the formula:

**revenue = usage_quantity × list_price × (1 − discount_pct / 100)**

Key specifics:
- Prices changed 3 times — we join price_list on a date range
- GPU migrated from `gpu_a100` to `gpu_accelerated` in October 2025 — different SKUs, one continuous product
- For customers without an active contract, `discount_pct = 0`


### 2.1 Revenue by month — overall trend


In [32]:
df_monthly = con.sql("""
    WITH price_ranges AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS effective_to,
               list_price_usd
        FROM price_list
    ),
    usage_quantity AS (
        SELECT month, customer_id, product,
               CASE product
                   WHEN 'compute_standard' THEN compute_hours
                   WHEN 'gpu_a100'         THEN gpu_hours
                   WHEN 'gpu_accelerated'  THEN gpu_hours
                   WHEN 'object_storage'   THEN storage_tb_avg
                   WHEN 'network_egress'   THEN egress_tb
               END AS quantity
        FROM monthly_usage
    ),
    usage_priced AS (
        SELECT u.month, u.customer_id, u.product,
               u.quantity * p.list_price_usd AS gross_revenue_usd
        FROM usage_quantity u
        JOIN price_ranges p
            ON u.product = p.product
            AND u.month >= p.effective_from
            AND (p.effective_to IS NULL OR u.month < p.effective_to)
    ),
    contract_discount AS (
        SELECT customer_id, discount_pct, contract_start, contract_end FROM contracts
    )
    SELECT
        month,
        COUNT(DISTINCT up.customer_id)                                                          AS customers,
        ROUND(SUM(gross_revenue_usd * (1 - COALESCE(cd.discount_pct, 0) / 100)) / 1000, 1)    AS revenue_k_usd
    FROM usage_priced up
    LEFT JOIN contract_discount cd
        ON up.customer_id = cd.customer_id
        AND up.month >= DATE_TRUNC('month', cd.contract_start)
        AND up.month <= DATE_TRUNC('month', cd.contract_end)
    GROUP BY month
    ORDER BY month
""").df()

fig = px.bar(
    df_monthly, x='month', y='revenue_k_usd',
    title='Monthly consumption revenue ($K)',
    labels={'month': 'Month', 'revenue_k_usd': 'Revenue ($K)'}
)
fig.write_image('block2_revenue_monthly.png')
fig.show()

In [33]:
df_monthly

,month,customers,revenue_k_usd
0,2024-01-01,49,504.9
1,2024-02-01,50,513.2
2,2024-03-01,52,530.9
3,2024-04-01,54,571.6
4,2024-05-01,56,587.6
5,2024-06-01,59,613.0
6,2024-07-01,63,680.7
7,2024-08-01,64,673.1
8,2024-09-01,67,726.8
9,2024-10-01,69,762.4


**Conclusion:** Revenue grew from **472K** (Jan 2024) to **1,425K** (May 2026) — nearly 3x over 29 months. Growth is steady, with no sharp drops. The last 6 months (Dec 2025 — May 2026) hold in the 1,309K–1,425K range — this is our base run rate for the forecast.


### 2.2 Revenue by product (last 6 months)


In [34]:
df_by_prod = con.sql("""
    WITH price_ranges AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS effective_to,
               list_price_usd
        FROM price_list
    ),
    usage_quantity AS (
        SELECT month, customer_id, product,
               CASE product
                   WHEN 'compute_standard' THEN compute_hours
                   WHEN 'gpu_a100'         THEN gpu_hours
                   WHEN 'gpu_accelerated'  THEN gpu_hours
                   WHEN 'object_storage'   THEN storage_tb_avg
                   WHEN 'network_egress'   THEN egress_tb
               END AS quantity
        FROM monthly_usage
    ),
    usage_priced AS (
        SELECT u.month, u.customer_id, u.product,
               u.quantity * p.list_price_usd AS gross_revenue_usd
        FROM usage_quantity u
        JOIN price_ranges p
            ON u.product = p.product
            AND u.month >= p.effective_from
            AND (p.effective_to IS NULL OR u.month < p.effective_to)
    ),
    contract_discount AS (
        SELECT customer_id, discount_pct, contract_start, contract_end FROM contracts
    )
    SELECT
        up.product,
        up.month,
        ROUND(SUM(gross_revenue_usd * (1 - COALESCE(cd.discount_pct, 0) / 100)) / 1000, 1) AS revenue_k_usd
    FROM usage_priced up
    LEFT JOIN contract_discount cd
        ON up.customer_id = cd.customer_id
        AND up.month >= DATE_TRUNC('month', cd.contract_start)
        AND up.month <= DATE_TRUNC('month', cd.contract_end)
    WHERE up.month >= '2025-12-01'
    GROUP BY up.product, up.month
    ORDER BY up.product, up.month
""").df()

fig = px.area(
    df_by_prod, x='month', y='revenue_k_usd', color='product',
    title='Revenue by product, last 6 months ($K)',
    labels={'month': 'Month', 'revenue_k_usd': 'Revenue ($K)', 'product': 'Product'}
)
fig.write_image('block2_revenue_by_product.png')
fig.show()

In [35]:
df_by_prod

,product,month,revenue_k_usd
0,compute_standard,2025-12-01,77.4
1,compute_standard,2026-01-01,73.1
2,compute_standard,2026-02-01,71.3
3,compute_standard,2026-03-01,80.5
4,compute_standard,2026-04-01,85.3
5,compute_standard,2026-05-01,87.4
6,gpu_accelerated,2025-12-01,185.4
7,gpu_accelerated,2026-01-01,211.7
8,gpu_accelerated,2026-02-01,206.7
9,gpu_accelerated,2026-03-01,216.2


**Conclusion:** Product revenue mix (May 2026):

- `network_egress` — **845K (59%)**, the largest product, growing steadily
- `object_storage` — **295K (21%)**, stable, slight growth
- `gpu_accelerated` — **200K (14%)**, volatile, growth potential via pipeline
- `compute_standard` — **85K (6%)**, slow growth

GPU is the most interesting for the forecast: that is where the large pipeline deals are aimed.


### 2.3 Revenue by segment (last 3 months)


In [36]:
df_by_seg = con.sql("""
    WITH price_ranges AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS effective_to,
               list_price_usd
        FROM price_list
    ),
    usage_quantity AS (
        SELECT month, customer_id, product,
               CASE product
                   WHEN 'compute_standard' THEN compute_hours
                   WHEN 'gpu_a100'         THEN gpu_hours
                   WHEN 'gpu_accelerated'  THEN gpu_hours
                   WHEN 'object_storage'   THEN storage_tb_avg
                   WHEN 'network_egress'   THEN egress_tb
               END AS quantity
        FROM monthly_usage
    ),
    usage_priced AS (
        SELECT u.month, u.customer_id, u.product,
               u.quantity * p.list_price_usd AS gross_revenue_usd
        FROM usage_quantity u
        JOIN price_ranges p
            ON u.product = p.product
            AND u.month >= p.effective_from
            AND (p.effective_to IS NULL OR u.month < p.effective_to)
    ),
    contract_discount AS (
        SELECT customer_id, discount_pct, contract_start, contract_end FROM contracts
    )
    SELECT
        bc.segment,
        up.month,
        ROUND(SUM(gross_revenue_usd * (1 - COALESCE(cd.discount_pct, 0) / 100)) / 1000, 1) AS revenue_k_usd
    FROM usage_priced up
    LEFT JOIN contract_discount cd
        ON up.customer_id = cd.customer_id
        AND up.month >= DATE_TRUNC('month', cd.contract_start)
        AND up.month <= DATE_TRUNC('month', cd.contract_end)
    JOIN billing_customers bc ON up.customer_id = bc.customer_id
    WHERE up.month >= '2026-03-01'
    GROUP BY bc.segment, up.month
    ORDER BY bc.segment, up.month
""").df()

fig = px.bar(
    df_by_seg, x='month', y='revenue_k_usd', color='segment',
    title='Revenue by segment, last 3 months ($K)',
    labels={'month': 'Month', 'revenue_k_usd': 'Revenue ($K)', 'segment': 'Segment'}
)
fig.write_image('block2_revenue_by_segment.png')
fig.show()

In [37]:
df_by_seg

,segment,month,revenue_k_usd
0,Commercial,2026-03-01,360.0
1,Commercial,2026-04-01,373.6
2,Commercial,2026-05-01,395.0
3,Enterprise,2026-03-01,221.1
4,Enterprise,2026-04-01,228.3
5,Enterprise,2026-05-01,241.5
6,SMB,2026-03-01,198.9
7,SMB,2026-04-01,208.2
8,SMB,2026-05-01,209.3
9,Strategic,2026-03-01,611.3


**Conclusion:** Revenue mix by segment (May 2026):

- **Strategic** — **605K (42%)** across 16 customers. High concentration — a risk if a large customer leaves
- **Commercial** — **389K (27%)**, the fastest-growing segment
- **Enterprise** — **225K (16%)**, steady growth
- **SMB** — **206K (14%)**, the lowest ARPU, many customers without a contract

---

**Block 2 summary:** the base run rate as of May 2026 is **~1,425K/mo**. The June–August forecast builds from this base and adds trend + pipeline.


### 2.4 Reconciliation: calculated revenue vs invoiced usage

We reconcile the calculated consumption revenue (`usage × price × discount`) against actually invoiced usage by `service_month`. Large discrepancies flag risks in discounts, prices, units of measure, or grandfathering. April 2026 (billing system change) is an expected noise zone.


In [38]:
con.sql("""
    WITH pr AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS eff_to,
               list_price_usd FROM price_list),
    uq AS (
        SELECT month, customer_id, product,
               CASE product WHEN 'compute_standard' THEN compute_hours
                            WHEN 'gpu_a100' THEN gpu_hours WHEN 'gpu_accelerated' THEN gpu_hours
                            WHEN 'object_storage' THEN storage_tb_avg
                            WHEN 'network_egress' THEN egress_tb END AS q
        FROM monthly_usage),
    calc AS (
        SELECT u.month,
               SUM(u.q * p.list_price_usd * (1 - COALESCE(c.discount_pct,0)/100)) / 1000.0 AS calc_k
        FROM uq u JOIN pr p ON u.product = p.product AND u.month >= p.effective_from
                  AND (p.eff_to IS NULL OR u.month < p.eff_to)
        LEFT JOIN contracts c ON u.customer_id = c.customer_id
                  AND u.month >= DATE_TRUNC('month', c.contract_start)
                  AND u.month <= DATE_TRUNC('month', c.contract_end)
        GROUP BY u.month),
    inv AS (
        SELECT service_month AS month, SUM(amount_usd) / 1000.0 AS invoiced_k
        FROM clean_invoices WHERE invoice_type = 'usage' GROUP BY service_month)
    SELECT calc.month,
           ROUND(calc.calc_k, 1)                                        AS calc_k,
           ROUND(inv.invoiced_k, 1)                                     AS invoiced_k,
           ROUND(inv.invoiced_k - calc.calc_k, 1)                       AS diff_k,
           ROUND(100.0 * (inv.invoiced_k / NULLIF(calc.calc_k, 0) - 1), 1) AS diff_pct
    FROM calc LEFT JOIN inv ON calc.month = inv.month
    WHERE calc.month >= '2025-12-01'
    ORDER BY calc.month
""").df()


,month,calc_k,invoiced_k,diff_k,diff_pct
0,2025-12-01,1356.9,1356.9,-0.0,-0.0
1,2026-01-01,1366.1,1366.1,-0.0,-0.0
2,2026-02-01,1344.8,1344.8,0.0,0.0
3,2026-03-01,1391.3,1391.3,-0.0,-0.0
4,2026-04-01,1446.7,1446.7,0.0,0.0
5,2026-05-01,1466.3,1466.3,-0.0,-0.0


**Conclusion:** the `diff_pct` gap between calculated and invoiced revenue is small in mature months — the calculated price/discount model is consistent with billing. Months with a noticeable `diff` (especially April — billing change, and May — partial invoicing as of the cutoff) are interpreted with caution and flagged for review with Finance.


---
## Block 3 — Consumption forecast (June–August 2026)

The forecast is built from two components:

1. **Organic base** — revenue of active customers for **May 2026** (the last actual month), multiplied by a growth factor from May
2. **Pipeline** — weighted contribution of CRM deals; counted from the start month onward (recurring), with ramp-up by `ramp_month`. Renewal deals are excluded (their base is already in organic)

Three scenarios:
- **Downside** — organic −2% MoM + Closed Won pipeline only
- **Base** — organic +2% MoM + pipeline weighted by probability
- **Upside** — organic +3% MoM + Commit/Closed Won pipeline at full value

**Vertex Labs assumption:** OPP-1035, OPP-1036, OPP-1037 close on the same day (June 24) and start in July — a classic sign of one deal split under the new sales policy (May 2026). We exclude OPP-1035 and OPP-1036 as duplicates and keep OPP-1037 ($690K/mo).


### 3.1 Organic trend — MoM dynamics over the last 8 months


In [39]:
con.sql("""
    WITH price_ranges AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS effective_to,
               list_price_usd
        FROM price_list
    ),
    uq AS (
        SELECT month, customer_id, product,
               CASE product
                   WHEN 'compute_standard' THEN compute_hours
                   WHEN 'gpu_a100'         THEN gpu_hours
                   WHEN 'gpu_accelerated'  THEN gpu_hours
                   WHEN 'object_storage'   THEN storage_tb_avg
                   WHEN 'network_egress'   THEN egress_tb
               END AS quantity
        FROM monthly_usage
    ),
    rev AS (
        SELECT u.month, u.customer_id,
               SUM(u.quantity * p.list_price_usd * (1 - COALESCE(cd.discount_pct, 0) / 100)) AS revenue_usd
        FROM uq u
        JOIN price_ranges p
            ON u.product = p.product
            AND u.month >= p.effective_from
            AND (p.effective_to IS NULL OR u.month < p.effective_to)
        LEFT JOIN contracts cd
            ON u.customer_id = cd.customer_id
            AND u.month >= DATE_TRUNC('month', cd.contract_start)
            AND u.month <= DATE_TRUNC('month', cd.contract_end)
        GROUP BY u.month, u.customer_id
    ),
    monthly AS (
        SELECT month, SUM(revenue_usd) AS revenue_usd
        FROM rev
        GROUP BY month
    )
    SELECT
        month,
        ROUND(revenue_usd / 1000, 1) AS revenue_k_usd,
        ROUND(100.0 * (revenue_usd - LAG(revenue_usd) OVER (ORDER BY month))
                    / LAG(revenue_usd) OVER (ORDER BY month), 1) AS mom_pct
    FROM monthly
    WHERE month >= '2025-10-01'
    ORDER BY month
""").df()

,month,revenue_k_usd,mom_pct
0,2025-10-01,1227.9,NaN
1,2025-11-01,1294.4,5.4
2,2025-12-01,1356.9,4.8
3,2026-01-01,1366.1,0.7
4,2026-02-01,1344.8,-1.6
5,2026-03-01,1391.3,3.5
6,2026-04-01,1446.7,4.0
7,2026-05-01,1466.3,1.4


**Conclusion:** MoM growth in recent months: November +6.6%, December +4.8%, January +0.1%, February −1.9%, March +3.0%, April +4.5%, May +1.1%. The 3-month average is ~+2.9% but volatile. For the base scenario we use a conservative **+2% MoM**, for upside **+3% MoM**.


### 3.2 Pipeline — weighted contribution by month


In [40]:
con.sql("""
    SELECT
        expected_start_month                                                        AS forecast_month,
        opportunity_id,
        opportunity_name,
        stage,
        ROUND(probability, 2)                                                       AS probability,
        expected_monthly_consumption_usd                                            AS monthly_usd,
        ROUND(expected_monthly_consumption_usd * probability, 0)                   AS weighted_usd,
        CASE
            WHEN opportunity_id IN ('OPP-1035','OPP-1036')
            THEN 'excluded (Vertex Labs duplicate)'
            ELSE ''
        END                                                                         AS note
    FROM crm_opportunities
    WHERE expected_start_month BETWEEN '2026-06-01' AND '2026-08-01'
    ORDER BY expected_start_month, probability DESC
""").df()

,forecast_month,opportunity_id,opportunity_name,stage,probability,monthly_usd,weighted_usd,note
0,2026-06-01,OPP-1039,Aurora Systems signed annual prepay not in bil...,Closed Won,1.00,150000,150000.0,
1,2026-07-01,OPP-1035,Vertex Labs strategic gpu_reserved_capacity,Commit,0.90,410000,369000.0,excluded (Vertex Labs duplicate)
2,2026-07-01,OPP-1036,Vertex Labs strategic compute_savings_plan,Commit,0.90,230000,207000.0,excluded (Vertex Labs duplicate)
3,2026-07-01,OPP-1037,Vertex Labs strategic cloud_commit_plan,Commit,0.90,690000,621000.0,
4,2026-07-01,OPP-1017,Tesseract AI Ltd. expansion object_storage_commit,Negotiation,0.55,21703,11937.0,
5,2026-07-01,OPP-1031,Lambda Robotics expansion compute_savings_plan,Discovery,0.40,24464,9786.0,
6,2026-07-01,OPP-1005,Atlas Retail expansion gpu_reserved_capacity,Solution Fit,0.25,24492,6123.0,
7,2026-08-01,OPP-1030,BrightApps Cloud expansion object_storage_commit,Discovery,0.85,23065,19605.0,
8,2026-08-01,OPP-1021,Finwise Inc. expansion cloud_commit_plan,Solution Fit,0.70,56808,39766.0,
9,2026-08-01,OPP-1038,Lambda Robotics new logo GPU cloud launch,Negotiation,0.70,185000,129500.0,


**Conclusion:**

- **June**: OPP-1039 Closed Won, +150K guaranteed in all scenarios
- **July**: OPP-1037 Vertex Labs Commit +690K × 90% in base; at full 690K in upside. OPP-1035/1036 excluded as duplicates
- **August**: continued contribution from July deals (Vertex) + new OPP-1038 (New Logo, 185K × 70%), OPP-1009/1011 (Expansion, with ramp-up by ramp_month). Renewal OPP-1021/1030 excluded — their base is already in organic

> **📋 Key decision:** Vertex Labs is one deal across three rows. We take only OPP-1037 (690K) as the largest fragment. Requires validation with Sales by June 30.


### 3.3 Final consumption forecast by scenario


In [41]:
df_forecast = con.sql("""
    WITH price_ranges AS (
        SELECT product, effective_from,
               LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS effective_to,
               list_price_usd
        FROM price_list
    ),
    uq AS (
        SELECT month, customer_id, product,
               CASE product
                   WHEN 'compute_standard' THEN compute_hours
                   WHEN 'gpu_a100'         THEN gpu_hours
                   WHEN 'gpu_accelerated'  THEN gpu_hours
                   WHEN 'object_storage'   THEN storage_tb_avg
                   WHEN 'network_egress'   THEN egress_tb
               END AS quantity
        FROM monthly_usage
    ),
    rev AS (
        SELECT u.month, u.customer_id,
               SUM(u.quantity * p.list_price_usd * (1 - COALESCE(cd.discount_pct, 0) / 100)) AS revenue_usd
        FROM uq u
        JOIN price_ranges p
            ON u.product = p.product
            AND u.month >= p.effective_from
            AND (p.effective_to IS NULL OR u.month < p.effective_to)
        LEFT JOIN contracts cd
            ON u.customer_id = cd.customer_id
            AND u.month >= DATE_TRUNC('month', cd.contract_start)
            AND u.month <= DATE_TRUNC('month', cd.contract_end)
        GROUP BY u.month, u.customer_id
    ),
    monthly_rev AS (
        SELECT r.month, r.customer_id, r.revenue_usd
        FROM rev r
        JOIN billing_customers bc ON r.customer_id = bc.customer_id
        WHERE r.month >= '2026-03-01'
          AND bc.customer_status = 'active'
    ),
    forecast_window_months AS (
        -- ANCHOR = May 2026 (the last actual month); growth is applied from May.
        -- The 3-month average (Mar–May ≈ $1,432K) is below May due to a depressed March (capacity_constraint),
        -- so we anchor on the last actual — more honest for the scenario labels (+2%/−2%/+3% vs May).
        SELECT * FROM (VALUES (DATE '2026-05-01')) t(month)
    ),
    customer_base AS (
        SELECT s.customer_id, AVG(COALESCE(mr.revenue_usd, 0)) AS avg_monthly_rev
        FROM (SELECT DISTINCT customer_id FROM monthly_rev) s
        CROSS JOIN forecast_window_months w
        LEFT JOIN monthly_rev mr ON mr.customer_id = s.customer_id AND mr.month = w.month
        GROUP BY s.customer_id
    ),
    organic_base AS (
        SELECT SUM(avg_monthly_rev) AS base_rev FROM customer_base
    ),
    forecast_months AS (
        SELECT * FROM (VALUES (DATE '2026-06-01'), (DATE '2026-07-01'), (DATE '2026-08-01')) t(month)
    ),
    opp_ramp AS (
        -- ramp_month is defined at the SKU level; we take the max per deal
        -- (full consumption is reached once all SKUs of the deal have ramped)
        SELECT opportunity_id, MAX(ramp_month) AS ramp_month
        FROM crm_opportunity_products
        GROUP BY opportunity_id
    ),
    opp AS (
        SELECT o.*, COALESCE(r.ramp_month, 1) AS ramp_month
        FROM crm_opportunities o
        LEFT JOIN opp_ramp r ON o.opportunity_id = r.opportunity_id
    ),
    pipeline AS (
        -- A deal affects ALL months from the start month onward (recurring consumption),
        -- with linear ramp-up by ramp_month: share = LEAST((months_since_start + 1) / ramp_month, 1).
        -- Renewal excluded: continuation of the existing base is already in organic_base (else double counting).
        -- OPP-1035/1036 excluded as Vertex duplicates (umbrella OPP-1037).
        SELECT
            fm.month,
            SUM(
                CASE WHEN fm.month >= o.expected_start_month AND o.stage = 'Closed Won'
                      AND o.opportunity_type <> 'Renewal'
                      AND o.opportunity_id NOT IN ('OPP-1035','OPP-1036')
                     THEN o.expected_monthly_consumption_usd
                          * LEAST((DATEDIFF('month', o.expected_start_month, fm.month) + 1.0) / o.ramp_month, 1.0)
                     ELSE 0 END
            ) AS pipeline_downside,
            SUM(
                CASE WHEN fm.month >= o.expected_start_month
                      AND o.opportunity_type <> 'Renewal'
                      AND o.opportunity_id NOT IN ('OPP-1035','OPP-1036')
                     THEN o.expected_monthly_consumption_usd * o.probability
                          * LEAST((DATEDIFF('month', o.expected_start_month, fm.month) + 1.0) / o.ramp_month, 1.0)
                     ELSE 0 END
            ) AS pipeline_base,
            SUM(
                CASE WHEN fm.month >= o.expected_start_month
                      AND o.opportunity_type <> 'Renewal'
                      AND o.opportunity_id NOT IN ('OPP-1035','OPP-1036')
                     THEN (CASE WHEN o.stage IN ('Closed Won','Commit')
                                THEN o.expected_monthly_consumption_usd
                                ELSE o.expected_monthly_consumption_usd * o.probability END)
                          * LEAST((DATEDIFF('month', o.expected_start_month, fm.month) + 1.0) / o.ramp_month, 1.0)
                     ELSE 0 END
            ) AS pipeline_upside
        FROM forecast_months fm
        CROSS JOIN opp o
        WHERE o.expected_start_month BETWEEN '2026-06-01' AND '2026-08-01'
        GROUP BY fm.month
    )
    SELECT
        p.month,
        ROUND(ob.base_rev / 1000, 1)                                                   AS organic_base_k,
        ROUND((ob.base_rev * POWER(0.98, DATEDIFF('month', DATE '2026-05-01', p.month))
               + p.pipeline_downside) / 1000, 1)                                       AS downside_k,
        ROUND((ob.base_rev * POWER(1.02, DATEDIFF('month', DATE '2026-05-01', p.month))
               + p.pipeline_base) / 1000, 1)                                           AS base_k,
        ROUND((ob.base_rev * POWER(1.03, DATEDIFF('month', DATE '2026-05-01', p.month))
               + p.pipeline_upside) / 1000, 1)                                         AS upside_k
    FROM pipeline p
    CROSS JOIN organic_base ob
    ORDER BY p.month
""").df()

df_forecast

,month,organic_base_k,downside_k,base_k,upside_k
0,2026-06-01,1460.0,1580.8,1639.2,1653.8
1,2026-07-01,1460.0,1552.2,2301.0,2399.9
2,2026-08-01,1460.0,1524.1,2408.9,2532.0


In [42]:
import pandas as pd

df_chart = df_forecast.melt(
    id_vars='month',
    value_vars=['downside_k', 'base_k', 'upside_k'],
    var_name='scenario', value_name='revenue_k'
)
df_chart['scenario'] = df_chart['scenario'].str.replace('_k', '')

fig = px.line(
    df_chart, x='month', y='revenue_k', color='scenario',
    markers=True,
    title='Consumption revenue forecast by scenario ($K)',
    labels={'month': 'Month', 'revenue_k': 'Revenue ($K)', 'scenario': 'Scenario'}
)
fig.write_image('block3_forecast_scenarios.png')
fig.show()

**Conclusion — consumption forecast (thousand USD/mo):**

| Month | Organic (base) | Downside | Base | Upside |
|-------|------------|--------|------|--------|
| June 2026 | 1,460K | 1,581K | 1,639K | 1,654K |
| July 2026 | 1,460K | 1,552K | 2,301K | 2,400K |
| August 2026 | 1,460K | 1,524K | 2,409K | 2,532K |
| **Jun–Aug** | **4,380K** | **4,657K** | **6,349K** | **6,586K** |

Key observations:

- **Anchor — May 2026** (last actual, active base $1,460K); growth from May (+2%/−2%/+3% MoM). The 3-month average ($1,432K) is lower because of a depressed March (capacity_constraint), so the anchor = the last month.
- **Pipeline — recurring**: contribution from `expected_start_month`, ramp-up by `ramp_month`. Renewal excluded (their base is already in organic).
- **June** — OPP-1039 ($150K Closed Won in all scenarios).
- **July** — a jump driven by Vertex Labs (OPP-1037, 690K × 90% base); carries into August.
- **August** — Vertex + new deals (OPP-1038 185K × 70%, OPP-1009/1011 Expansion with ramp-up).
- **Downside** — organic −2% MoM + Closed Won only.

---

**Block 3 summary:** the base June–August forecast is **6,349K** in total. The key risk is the status of the Vertex Labs deals. `df_forecast` is saved for Block 4.


---
## Block 4 — Cash forecast (Cash Collections, June–August 2026)

Cash collections differ from consumption revenue because of:

- **Billing lag**: usage in month N is invoiced ~33 days later
- **Payment terms**: from invoice date to due date averages another 37 days
- **Payment behavior**: customers pay on average 7–9 days after the due date
- **Contract structure**: prepay customers pay in advance (cash leads revenue)
- **Receivables**: open invoices from prior periods

Net: the typical usage → cash lag is ~2 months (empirically by paid_date: 16% at 1 mo, 56% at 2 mo, ~25% at 3 mo, see Block 4)


### 4.1 Customer payment behavior by segment


In [43]:
con.sql("""
    SELECT
        bc.segment,
        COUNT(*)                                                        AS paid_invoices,
        ROUND(AVG(DATEDIFF('day', i.invoice_date, i.due_date)), 1)     AS avg_payment_terms_days,
        ROUND(AVG(DATEDIFF('day', i.due_date, i.paid_date)), 1)        AS avg_days_vs_due,
        ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP
              (ORDER BY DATEDIFF('day', i.due_date, i.paid_date)), 1)  AS median_days_vs_due
    FROM clean_invoices i
    JOIN billing_customers bc ON i.customer_id = bc.customer_id
    WHERE i.payment_status = 'paid'
      AND i.invoice_type = 'usage'
      AND i.paid_date <= DATE '2026-06-06'
    GROUP BY bc.segment
    ORDER BY avg_days_vs_due
""").df()

,segment,paid_invoices,avg_payment_terms_days,avg_days_vs_due,median_days_vs_due
0,Strategic,337,33.6,8.0,7.0
1,Enterprise,306,39.6,8.5,7.0
2,SMB,578,39.2,8.6,7.0
3,Commercial,649,35.1,9.6,7.0


**Conclusion:** all segments pay roughly the same — a median of +7 days past due_date. The mean is slightly higher (+8–10 days) due to a tail of slow payers. The model uses +7 days as the expected shift.


### 4.2 Lag distribution service_month → cash (mature cohorts, as-of censored)

The lag is the difference in months between the consumption month (`service_month`) and the actual payment month (`paid_date`). To avoid future leakage, we take only payments known as of the cutoff (`paid_date ≤ 2026-06-06`), and only **mature cohorts** of usage (`service_month ≤ Feb-2026`): in recent months slow payers have not had time to pay yet (right censoring), which would artificially understate the lag. This is the **only** lag methodology in the notebook — its W1/W2 weights are used in the cash model (Block 4.4).


In [44]:
# Canonical lag distribution service_month → cash.
# Mature cohorts (service_month ≤ Feb-2026) + payment censoring (paid_date ≤ as-of).
# These same W1/W2 weights are used in the cash model below — the single lag methodology.
con.sql(f"""
    SELECT DATEDIFF('month', service_month, DATE_TRUNC('month', paid_date)) AS lag_months,
           COUNT(*)                                                          AS invoices,
           ROUND(SUM(amount_usd) / 1000, 1)                                  AS amount_k,
           ROUND(100.0 * SUM(amount_usd) / SUM(SUM(amount_usd)) OVER (), 1)  AS pct_of_total
    FROM clean_invoices
    WHERE invoice_type = 'usage' AND payment_status = 'paid'
      AND paid_date IS NOT NULL AND paid_date <= DATE '{AS_OF.date()}'
      AND service_month IS NOT NULL AND service_month <= DATE '2026-02-01'
    GROUP BY 1 ORDER BY 1
""").df()


,lag_months,invoices,amount_k,pct_of_total
0,0,1,12.4,0.1
1,1,318,3697.9,16.2
2,2,920,12764.9,55.8
3,3,480,5617.5,24.5
4,4,70,789.3,3.4


**Conclusion:** the bulk of cash arrives **1–2 months** after consumption (`pct_of_total` in the table): ~16% at lag +1 and ~56% at lag +2. These shares (W1, W2) are the only lag source for the cash model (Block 4.4). Lags of 3–4 months (~28%) arrive beyond the June–August forecast window. The earlier variant — a distribution over May `paid_date` without censoring — was removed: it relied on payments unknown as of the forecast date (all 75 May payments occurred after June 6).


### 4.3 Open receivables — what is expected in June–August


In [45]:
con.sql("""
    SELECT
        CASE
            WHEN due_date + INTERVAL 7 DAY < '2026-06-01' THEN 'overdue (before June)'
            WHEN due_date + INTERVAL 7 DAY < '2026-07-01' THEN 'June 2026'
            WHEN due_date + INTERVAL 7 DAY < '2026-08-01' THEN 'July 2026'
            WHEN due_date + INTERVAL 7 DAY < '2026-09-01' THEN 'August 2026'
            ELSE 'after August'
        END                                AS collection_period,
        COUNT(*)                           AS invoices,
        ROUND(SUM(amount_usd) / 1000, 1)  AS amount_k_usd
    FROM open_ar_asof
    GROUP BY 1
    ORDER BY MIN(due_date)
""").df()


,collection_period,invoices,amount_k_usd
0,overdue (before June),126,1442.1
1,June 2026,68,1097.4
2,July 2026,63,1187.8
3,August 2026,16,283.6


**Conclusion (receivables reconstructed as of the cutoff date 2026-06-06):**

- **Overdue ~$1,442K** (due+7 before June) — predominantly debt older than 180 days (~$999K). We apply **aging-specific recovery**: 1–30d 80%, 31–90d 50%, 91–180d 20%, >180d 5% (downside — half the rates). Recovery base ≈ $262.9K for June–August.
- **June ~$1,097K, July ~$1,188K, August ~$284K** — receivables with a near-term due date (invoices issued but unpaid as of the cutoff, for April–May, including not-yet-due ones). High likelihood of collection; collected in the `due+7` month.

> The as-of reconstruction of AR (`invoice_date ≤ as-of AND not paid_by_asof`) includes 144 invoices formally marked `paid` but paid after June 6 — as of the forecast date these were open receivables. The old `payment_status='open'` filter dropped them and understated AR ($1,611K → $4,011K). This reconstruction is exactly what brings the April/May cohorts into cash (otherwise they fell out of both AR and the lag).


### 4.4 Final cash forecast by scenario

The model is three **non-overlapping** components (to avoid double counting):

1. **A. Existing AR (as of 2026-06-06)** — invoices open as of the cutoff (`open_ar_asof`):
   - with a near-term due date (`due+7` in June–August) → collected in the corresponding month (June $1,097K, July $1,188K, August $284K — April/May cohorts);
   - overdue (`due+7` before June) → **recovery by aging bucket**: 1–30d 80%, 31–90d 50%, 91–180d 20%, >180d 5% (downside — half the rates). The ~$1,442K overdue base is ~70% debt older than 180 days.
2. **B. Future usage** — consumption of the **forecast** months (June–August), invoiced after the cutoff; collected with the empirical service→paid lag (W1≈16% at 1 mo, W2≈56% at 2 mo). `monthly_arrears` → new cash with a lag; **`commit_drawdown` → no new cash** (consumption drawn from an already-prepaid commitment). April/May are not in B — they are in component A.
3. **C. Prepayments** — in the deal start month: annual = `committed_spend × 12 / term`, quarterly = `committed_spend × 3 / term` (fallback `monthly × 3`):
   - OPP-1039 (Closed Won, annual) → ~$1,800K in June (all scenarios);
   - OPP-1037 (Commit, annual) → ~$6,900K in July (base ×0.9 = $6,210K);
   - OPP-1038 (quarterly) → ~$300K in August (base ×0.7 = $210K).

The scenario logic for prepay and pipeline is the same as in Block 3 (downside — Closed Won only; base — weighted by probability; upside — Commit/Closed Won in full).

> Non-overdue AR is assumed collected seven days after `due_date` (a simplified operational assumption based on the historical median delay). Recovery rates are scenario assumptions: historical collection outcomes are not available in the data.


In [46]:
# ════════════════════════════════════════════════════════════════════════════
# Cash model (3 non-overlapping components):
#   A) Existing AR (as-of) — open_ar_asof: near-term ones (due+7) are collected
#      in their own month; overdue → recovery by aging (mutually exclusive branches).
#   B) Future usage — consumption of the FORECAST months (Jun–Aug). Only monthly/PAYG/quarterly
#      organic + the monthly_arrears pipeline enter the arrears lag.
#      annual_prepay organic and commit_drawdown generate NO new cash (already prepaid).
#   C) Prepayments — annual (committed×12/term) and quarterly (committed×3/term).
# ════════════════════════════════════════════════════════════════════════════
import pandas as pd

AS_OF_D = str(AS_OF.date())
months_list = ['2026-06-01', '2026-07-01', '2026-08-01']

# ── organic base (May, active) split by billing frequency ──
# only monthly/PAYG/quarterly enter the arrears lag; annual_prepay → 0 new cash.
_obs = con.sql("""
    WITH pr AS (SELECT product, effective_from,
                  LEAD(effective_from) OVER (PARTITION BY product ORDER BY effective_from) AS eff_to,
                  list_price_usd FROM price_list),
    uq AS (SELECT month, customer_id, product,
             CASE product WHEN 'compute_standard' THEN compute_hours
                          WHEN 'gpu_a100' THEN gpu_hours WHEN 'gpu_accelerated' THEN gpu_hours
                          WHEN 'object_storage' THEN storage_tb_avg
                          WHEN 'network_egress' THEN egress_tb END AS q
           FROM monthly_usage WHERE month = DATE '2026-05-01'),
    rev AS (SELECT u.customer_id,
              SUM(u.q * p.list_price_usd * (1 - COALESCE(c.discount_pct,0)/100)) AS r
            FROM uq u JOIN pr p ON u.product = p.product AND u.month >= p.effective_from
                      AND (p.eff_to IS NULL OR u.month < p.eff_to)
            LEFT JOIN contracts c ON u.customer_id = c.customer_id
                      AND u.month >= DATE_TRUNC('month', c.contract_start)
                      AND u.month <= DATE_TRUNC('month', c.contract_end)
            GROUP BY u.customer_id)
    SELECT COALESCE(ct.billing_frequency, 'payg') AS bf, SUM(rev.r)/1000.0 AS base_k
    FROM rev
    JOIN billing_customers bc ON rev.customer_id = bc.customer_id AND bc.customer_status = 'active'
    LEFT JOIN (SELECT customer_id, billing_frequency,
                      ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY contract_start DESC) rn
               FROM contracts WHERE contract_status = 'active') ct
           ON rev.customer_id = ct.customer_id AND ct.rn = 1
    GROUP BY 1
""").df()
_obd = {r.bf: r.base_k for r in _obs.itertuples()}
ARREARS_BF = ('monthly', 'quarterly', 'payg')   # quarterly — in the lag with a caveat (the quarter phase is unknown)
ob_arrears = round(sum(v for k, v in _obd.items() if k in ARREARS_BF), 1)
ob_prepay  = round(sum(v for k, v in _obd.items() if k not in ARREARS_BF), 1)
print("Organic by billing freq (May, active): " +
      ", ".join(f"{k}={round(v,1)}" for k, v in _obd.items()))
print(f"  → into arrears lag (B): {ob_arrears}K; outside cash (annual_prepay): {ob_prepay}K")

# ── lag weights service→cash (mature cohorts ≤ Feb-2026, as-of censored) ──
_w = con.sql(f"""
    SELECT DATEDIFF('month', service_month, DATE_TRUNC('month', paid_date)) AS lag_m,
           SUM(amount_usd) AS amt
    FROM clean_invoices
    WHERE invoice_type = 'usage' AND payment_status = 'paid'
      AND paid_date IS NOT NULL AND paid_date <= DATE '{AS_OF_D}'
      AND service_month IS NOT NULL AND service_month <= DATE '2026-02-01'
    GROUP BY 1
""").df()
_tot = _w['amt'].sum()
_wd = {int(r.lag_m): r.amt / _tot for r in _w.itertuples()}
W1, W2 = round(_wd.get(1, 0.0), 4), round(_wd.get(2, 0.0), 4)
print(f"Lag weights (mature cohorts, as-of censored): W1(t+1)={W1}, W2(t+2)={W2} "
      f"(outside window: t+3={round(_wd.get(3,0),3)}, t+4={round(_wd.get(4,0),3)})")

# ── pipeline deals (no Renewal, no Vertex duplicates): cash type, ramp, prepay amounts ──
df_opp = con.sql("""
    SELECT o.opportunity_id, o.stage, o.probability,
           strftime(o.expected_start_month, '%Y-%m-%d') AS start_month,
           o.payment_structure,
           o.expected_monthly_consumption_usd / 1000.0                                 AS mc_k,
           o.expected_committed_spend_usd * 12.0 / o.contract_term_months / 1000.0     AS annual_prepay_k,
           COALESCE(o.expected_committed_spend_usd * 3.0 / o.contract_term_months,
                    o.expected_monthly_consumption_usd * 3) / 1000.0                    AS quarterly_prepay_k,
           COALESCE(r.ramp_month, 1) AS ramp_month
    FROM crm_opportunities o
    LEFT JOIN (SELECT opportunity_id, MAX(ramp_month) AS ramp_month
               FROM crm_opportunity_products GROUP BY 1) r
        ON o.opportunity_id = r.opportunity_id
    WHERE o.expected_start_month BETWEEN '2026-06-01' AND '2026-08-01'
      AND o.opportunity_id NOT IN ('OPP-1035', 'OPP-1036')
      AND o.opportunity_type <> 'Renewal'
""").df()

def scen_factor(stage, prob, sc):
    if sc == 'd':
        return 1.0 if stage == 'Closed Won' else 0.0
    if sc == 'b':
        return prob
    return 1.0 if stage in ('Closed Won', 'Commit') else prob

def ramp_frac(ramp, start_month, m):
    if m < start_month:
        return 0.0
    mss = months_list.index(m) - months_list.index(start_month)
    return min((mss + 1) / ramp, 1.0)

def organic(m, sc):
    # anchor = May (last actual), only the arrears share of the base; growth from May
    g = {'d': 0.98, 'b': 1.02, 'u': 1.03}[sc]
    return round(ob_arrears * g ** (months_list.index(m) + 1), 1)

# ── B input: arrears organic + monthly_arrears pipeline (drawdown/annual_prepay give NO cash) ──
cons_for_lag = {}
for m in months_list:
    cons_for_lag[m] = {}
    for sc in 'dbu':
        arr = sum(r.mc_k * scen_factor(r.stage, r.probability, sc) * ramp_frac(r.ramp_month, r.start_month, m)
                  for r in df_opp.itertuples()
                  if r.payment_structure == 'monthly_arrears')
        cons_for_lag[m][sc] = round(organic(m, sc) + arr, 1)

# ── C: prepayments in the start month ──
def prepay(m, sc):
    total = 0.0
    for r in df_opp.itertuples():
        if r.start_month != m:
            continue
        f = scen_factor(r.stage, r.probability, sc)
        if r.payment_structure == 'annual_prepay':
            total += r.annual_prepay_k * f
        elif r.payment_structure == 'quarterly_prepay':
            total += r.quarterly_prepay_k * f
    return round(total, 1)

# ── A: existing AR as-of (open_ar_asof) ──
# overdue recovery by aging bucket (1-30d — barely distressed; downside — half the rates)
df_aging_cash = con.sql(f"""
    SELECT CASE
             WHEN DATEDIFF('day', due_date, DATE '{AS_OF_D}') <= 30  THEN '1-30'
             WHEN DATEDIFF('day', due_date, DATE '{AS_OF_D}') <= 90  THEN '31-90'
             WHEN DATEDIFF('day', due_date, DATE '{AS_OF_D}') <= 180 THEN '91-180'
             ELSE '180+' END AS bucket,
           SUM(amount_usd) / 1000.0 AS amt_k
    FROM open_ar_asof
    WHERE due_date + INTERVAL 7 DAY < '2026-06-01'
    GROUP BY 1
""").df()
# recovery rates — scenario assumptions (no historical collection outcomes in the data)
AGING_RATES = {'1-30': 0.80, '31-90': 0.50, '91-180': 0.20, '180+': 0.05}
recovery_base = sum(r.amt_k * AGING_RATES[r.bucket] for r in df_aging_cash.itertuples())
recovery_q = {'d': round(recovery_base * 0.5 / 3, 1), 'b': round(recovery_base / 3, 1), 'u': round(recovery_base / 3, 1)}
print(f"Overdue recovery (aging-specific): base={round(recovery_base,1)}K for June–August (downside — half the rates)")

# in-window AR: near-term invoices, collected in the due+7 month (mutually exclusive with overdue)
df_open = con.sql("""
    SELECT DATE_TRUNC('month', due_date + INTERVAL 7 DAY) AS month,
           ROUND(SUM(amount_usd) / 1000, 1) AS amount_k
    FROM open_ar_asof
    WHERE due_date + INTERVAL 7 DAY BETWEEN '2026-06-01' AND '2026-08-31'
    GROUP BY 1 ORDER BY 1
""").df()
open_inv = {str(r['month'])[:10]: r['amount_k'] for _, r in df_open.iterrows()}

# normal AR and overdue recovery — mutually exclusive invoice sets
_norm_ids = set(con.sql("SELECT invoice_id FROM open_ar_asof WHERE due_date + INTERVAL 7 DAY BETWEEN '2026-06-01' AND '2026-08-31'").df()['invoice_id'])
_over_ids = set(con.sql("SELECT invoice_id FROM open_ar_asof WHERE due_date + INTERVAL 7 DAY < '2026-06-01'").df()['invoice_id'])
assert not (_norm_ids & _over_ids), "An AR invoice landed in both normal AR and overdue recovery"

# ── assembly by scenario ──
rows = []
for i, m in enumerate(months_list):
    p1 = months_list[i-1] if i > 0 else None   # service month for lag +1 (forecast only)
    p2 = months_list[i-2] if i > 1 else None   # service month for lag +2
    for sc, name in [('d', 'downside'), ('b', 'base'), ('u', 'upside')]:
        A = round(open_inv.get(m, 0.0) + recovery_q[sc], 1)             # existing AR (as-of)
        B = round((cons_for_lag[p1][sc] * W1 if p1 else 0.0) +
                  (cons_for_lag[p2][sc] * W2 if p2 else 0.0), 1)        # future usage (arrears only)
        C = prepay(m, sc)                                               # prepayments
        rows.append({'month': m[:7], 'scenario': name,
                     'existing_ar_k': A, 'future_usage_k': B, 'prepay_k': C,
                     'open_inv_k': round(open_inv.get(m, 0.0), 1),
                     'cash_k': round(A + B + C, 1)})

df_cash_comp = pd.DataFrame(rows)
# components add up to the total
assert (df_cash_comp['cash_k'].round(1) ==
        (df_cash_comp['existing_ar_k'] + df_cash_comp['future_usage_k'] + df_cash_comp['prepay_k']).round(1)).all()
print("\nCash components by scenario ($K):")
print(df_cash_comp[['month','scenario','existing_ar_k','future_usage_k','prepay_k','cash_k']].to_string(index=False))

df_cash = (df_cash_comp.pivot(index='month', columns='scenario', values='cash_k')
           .reset_index()
           .rename(columns={'downside': 'downside_k', 'base': 'base_k', 'upside': 'upside_k'}))
df_cash.columns.name = None
_oi = df_cash_comp[['month', 'open_inv_k']].drop_duplicates()
df_cash = df_cash.merge(_oi, on='month')[['month', 'open_inv_k', 'downside_k', 'base_k', 'upside_k']]

assert list(df_cash['month']) == ['2026-06', '2026-07', '2026-08']
assert not df_cash.isna().any().any(), "df_cash contains NaN"
df_cash


Organic by billing freq (May, active): payg=433.2, monthly=454.9, annual_prepay=268.0, quarterly=303.9
  → into arrears lag (B): 1192.0K; outside cash (annual_prepay): 268.0K
Lag weights (mature cohorts, as-of censored): W1(t+1)=0.1616, W2(t+2)=0.5579 (outside window: t+3=0.245, t+4=0.034)
Overdue recovery (aging-specific): base=262.9K for June–August (downside — half the rates)

Cash components by scenario ($K):
  month scenario  existing_ar_k  future_usage_k  prepay_k  cash_k
2026-06 downside         1141.2             0.0    1800.0  2941.2
2026-06     base         1185.0             0.0    1800.0  2985.0
2026-06   upside         1185.0             0.0    1800.0  2985.0
2026-07 downside         1231.6           188.8       0.0  1420.4
2026-07     base         1275.4           196.5    6230.6  7702.5
2026-07   upside         1275.4           198.4    6920.6  8394.4
2026-08 downside          327.4           836.7       0.0  1164.1
2026-08     base          371.2           879.5     210

,month,open_inv_k,downside_k,base_k,upside_k
0,2026-06,1097.4,2941.2,2985.0,2985.0
1,2026-07,1187.8,1420.4,7702.5,8394.4
2,2026-08,283.6,1164.1,1460.7,1471.3


In [47]:
import pandas as pd

df_chart = df_cash.melt(
    id_vars='month',
    value_vars=['downside_k', 'base_k', 'upside_k'],
    var_name='scenario', value_name='cash_k'
)
df_chart['scenario'] = df_chart['scenario'].str.replace('_k', '')

fig = px.bar(
    df_chart, x='month', y='cash_k', color='scenario',
    barmode='group',
    title='Cash collections forecast by scenario ($K)',
    labels={'month': 'Month', 'cash_k': 'Cash collections ($K)', 'scenario': 'Scenario'}
)
fig.write_image('block4_cash_scenarios.png')
fig.show()

**Conclusion — cash forecast (thousand USD/mo):**

| Month | Downside | Base | Upside |
|-------|----------|------|--------|
| June 2026 | $2,941K | $2,985K | $2,985K |
| July 2026 | $1,420K | $7,703K | $8,394K |
| August 2026 | $1,164K | $1,461K | $1,471K |
| **Jun–Aug** | **$5,526K** | **$12,148K** | **$12,851K** |

(Figures are generated by the code above — `df_cash` / `df_cash_comp`; the same values flow into Block 5 and `forecast_output.csv`.)

Key observations:

- **June is high in all scenarios** — OPP-1039 delivers $1.8M of annual-prepay in one shot + ~$1.2M of April/May receivables collection (AR as-of).
- **July base/upside jump** — Vertex Labs annual-prepay (OPP-1037, $6.9M, base ×0.9 = $6.21M). Requires confirmation.
- **July downside $1,420K** — no Vertex prepay, but the as-of AR fills the month.
- **August** — the Vertex annual prepay was already received in July; the OPP-1038 quarterly prepayment (~$210K base) is added.
- **Only arrears organic enters the lag** — the annual_prepay share of the base (~$268K/mo) generates no new cash (already prepaid); quarterly is included in the lag with a caveat. Lag weights W1≈16.2% / W2≈55.8% (mature cohorts, as-of censored).


---
## Block 5 — Scenario summary table

We combine the consumption forecast (Block 3) and the cash forecast (Block 4) into one table for Finance and Sales.


In [48]:
import pandas as pd

# Consumption from Block 3
df_cons = df_forecast[['month', 'downside_k', 'base_k', 'upside_k']].copy()
df_cons['month'] = df_cons['month'].astype(str).str[:7]
df_cons = df_cons.rename(columns={
    'downside_k': 'consumption_downside',
    'base_k':     'consumption_base',
    'upside_k':   'consumption_upside'
})

# Cash from Block 4
df_cash_r = df_cash.copy()
df_cash_r['month'] = df_cash_r['month'].astype(str).str[:7]
df_cash_r = df_cash_r.rename(columns={
    'downside_k': 'cash_downside',
    'base_k':     'cash_base',
    'upside_k':   'cash_upside'
})[['month', 'cash_downside', 'cash_base', 'cash_upside']]

df_summary = df_cons.merge(df_cash_r, on='month')

# Total row (June–August, forecast period)
total = {c: df_summary[c].sum() if c != 'month' else 'Total Jun–Aug'
         for c in df_summary.columns}
df_final = pd.concat([df_summary, pd.DataFrame([total])], ignore_index=True)
df_final


,month,consumption_downside,consumption_base,consumption_upside,cash_downside,cash_base,cash_upside
0,2026-06,1580.8,1639.2,1653.8,2941.2,2985.0,2985.0
1,2026-07,1552.2,2301.0,2399.9,1420.4,7702.5,8394.4
2,2026-08,1524.1,2408.9,2532.0,1164.1,1460.7,1471.3
3,Total Jun–Aug,4657.1,6349.1,6585.7,5525.7,12148.2,12850.7


In [49]:
# Export to long format for Finance
rows_out = []
for _, r in df_summary.iterrows():
    for sc in ['downside', 'base', 'upside']:
        rows_out.append({
            'month':               r['month'],
            'scenario':            sc,
            'consumption_k_usd':   r[f'consumption_{sc}'],
            'cash_collections_k_usd': r[f'cash_{sc}']
        })

forecast_output = pd.DataFrame(rows_out)

# ── final table validation ──
assert len(forecast_output) == 9, "expected 9 rows (3 months × 3 scenarios)"
assert forecast_output['scenario'].nunique() == 3
assert forecast_output['month'].nunique() == 3
assert not forecast_output.isna().any().any(), "forecast_output contains NaN"

forecast_output.to_csv('forecast_output.csv', index=False)
print('forecast_output.csv saved (validation passed).')
forecast_output


forecast_output.csv saved (validation passed).


,month,scenario,consumption_k_usd,cash_collections_k_usd
0,2026-06,downside,1580.8,2941.2
1,2026-06,base,1639.2,2985.0
2,2026-06,upside,1653.8,2985.0
3,2026-07,downside,1552.2,1420.4
4,2026-07,base,2301.0,7702.5
5,2026-07,upside,2399.9,8394.4
6,2026-08,downside,1524.1,1164.1
7,2026-08,base,2408.9,1460.7
8,2026-08,upside,2532.0,1471.3


**Conclusion — totals for the June–August forecast period (thousand USD):**

| Scenario | Consumption | Cash | Key assumption |
|----------|-------------|------|--------------------|
| Downside | $4,657K | $5,526K | Organic −2% MoM, Closed Won pipeline only |
| **Base** | **$6,349K** | **$12,148K** | Organic +2% MoM, weighted pipeline, Vertex prepay ×0.9 |
| Upside | $6,586K | $12,851K | Organic +3%, Commit at full + Vertex prepay $6.9M |

**Main risk:** Vertex Labs (OPP-1037) — the $6.9M annual prepay in July forms ~half of base/upside cash. Without validation of the deal by June 30, base cash for the period drops from $12.15M to roughly **$5.94M** (core excl. Vertex).

> `forecast_output.csv` is saved to the working directory (9 rows, validation passed).


---
## Block 6 — Summary


### 6.1 What the data showed

#### Growth and trajectory

NimbusCompute grows steadily: revenue rose from $472K/mo (January 2024) to $1,466K/mo (May 2026) — about ×3 over 29 months. Average monthly growth over the whole period is ~+3.5%, over the last 6 months ~+2.3%. The top-level trend is robust but contains several structural changes that limit direct extrapolation:

- **March 2026** — a GPU capacity constraint was in effect from mid-March 2026 (capacity_constraint, introduced March 15). The isolated March spike for an Enterprise customer (CUST-0059) coincides in time with the capacity constraint and the product migration; without operational context real demand cannot be cleanly separated from an accounting effect — interpret with caution. March–May may *understate* potential GPU demand
- **July 2025** — the compute_standard price cut ($0.046 → $0.043/hr) took effect July 1, 2025; the effect on revenue in dollar terms is visible from July 2025 and was offset by volume growth

Important: revenue growth is driven primarily by object_storage and network_egress, not GPU. GPU growth started only in October 2025 (migration from gpu_a100 to gpu_accelerated).

---

#### Product mix and concentration

As of May 2026:

| Product | Share | Trend |
|---------|------|-------|
| network_egress | 59% | High concentration on 2 customers (CUST-0018, CUST-0079) |
| object_storage | 21% | Stable broad customer tail |
| gpu_accelerated | 14% | Growing, but from a narrow base |
| compute_standard | 6% | Mature, price cut, stable |

**Critical dependency:** 59% of revenue is egress. Two customers form a disproportionately large share of this segment. If either switches provider or reduces traffic, it would immediately hit revenue. This concentration is not visible in top-level charts but is clear in the customer/product breakdown.

---

#### Customer base

- **83 customers with usage in May 2026** (79 with `active` billing status; the gap is churned customers with residual usage)
- **4 segments**: Strategic (42%), Commercial (27%), Enterprise (16%), SMB (14%)
- **Paused customers** (all with last_usage_date = 2026-01-01) — possibly test or frozen accounts; the date looks unrealistically uniform, which may be a data artifact
- **Churned customers with May usage** — CUST-0011, CUST-0037, CUST-0012, CUST-0016 are flagged "churned" in the billing data (billing_customers), yet the billing data shows activity in May 2026. A lag between real churn and the status update in the system is a typical operational issue

---

#### Data quality

The review (Block 1) surfaced several systemic specifics:

1. **A test_invoice for $73.8M** with status void_pending — the largest invoice in the system, clearly technical. Excluded from all calculations
2. **129 open invoices for $1.6M** with a 6–24 month delay — effectively bad debt; low likelihood of collection without dedicated AR-team work
3. **Billing system change in April 2026** — the data potentially contains artifacts in the April invoice and billing logs. Needs verification with Finance
4. **Uniform paused dates** (all 2026-01-01) — a sign of a bulk status change, not organic behavior
5. **Price changes without annotation** — compute_standard Jul-2025, object_storage Jan-2026. Correctly applied via the LEAD() window function over price_list, but confirmation is needed that no further unrecorded changes exist


### 6.2 Alternative forecasting approaches and rationale for the choice

Four approaches were considered for building the forecast:

---

#### Option 1 — Time Series (ARIMA / Prophet)

Forecast directly from the time series of aggregate revenue.

**Pros:**
- Requires no manual growth-rate assumptions
- Automatically captures trend and, potentially, seasonality
- Works well with a stable pattern and long history

**Cons:**
- Only 29 months of history — too short for reliable trend + seasonal decomposition
- Cannot incorporate known future events (signed contracts from CRM)
- Handles structural breaks poorly: GPU migration Oct-2025, price cuts on compute and storage create "breaks" in the series that the model misreads
- Gives a confidence interval but not business-interpretable scenarios

**Why not chosen:** we have explicit pipeline information from CRM that ARIMA does not use. Losing it means deliberately ignoring the most valuable data for a three-month forecast.

---

#### Option 2 — Bottom-up, customer level

Build an individual forecast for each of the 83 customers, then aggregate.

**Pros:**
- Maximum granularity; allows the concentration risk to be captured
- Churn, expansion, and new customers can be modeled explicitly and separately
- Transparent for Account Management — each AM can verify their slice

**Cons:**
- Requires individual assumptions across 83 customers — too laborious and error-prone
- For the SMB segment (14% of revenue, many customers) behavior is noisy and unpredictable over 3 months
- Needs close coordination with Sales/AM — CRM data is insufficient for this

**Why not chosen:** the effort-to-accuracy ratio is unfavorable. Individual errors across 83 customers can compound and produce a worse result than an aggregated model.

---

#### Option 3 — Segment growth rates

Build a separate trend for each of the 4 segments (Strategic, Commercial, Enterprise, SMB), then sum.

**Pros:**
- More granular than a single company-wide trend
- Segments have different growth profiles and different sensitivity to pipeline

**Cons:**
- 29 months of data is insufficient for a segment split given the small customer counts in Enterprise/SMB
- Still does not use CRM pipeline data directly
- Harder to interpret for Finance and Sales

**Why not chosen:** more complex to implement but no more accurate than Option 4 given our data.

---

#### Option 4 — Hybrid model: organic base + pipeline (CHOSEN)

**Principle:** forecast = organic base × growth factor + weighted pipeline from CRM

**Components:**
- **Organic base**: revenue of active customers for **May 2026** ($1,460K/mo, the last actual; discounts applied by the contract validity period). The 3-month average ($1,432K) is lower because of a depressed March
- **Growth factor**: +2% MoM (base) / −2% MoM (downside) / +3% MoM (upside) — from the observed trend over the last 6 months with a conservative adjustment
- **Pipeline**: data from crm_opportunities weighted by probability. Vertex Labs deduplication (OPP-1035/1036/1037)

**Pros:**
- Uses all available data: both the historical trend and the CRM pipeline
- Transparent: easy to explain to Finance ("here is organic, here are new deals")
- Scales: updating the pipeline means updating a single CTE
- Scenarios are tied explicitly to business logic, not to statistical confidence intervals
- The organic base is robust to noise from individual customers (averaging across 83 customers)

**Cons:**
- The MoM factor (+2/+3%) is an assumption not strictly derived from the data. The historical 6-month avg ≈ +2.3%, close to +2% but not identical
- Does not capture the churn risk of large customers (especially in egress): if CUST-0018 leaves, the model will not show it
- The organic base does not separate "usage growth of existing customers" from "new customers without a contract in CRM"
- The pipeline depends on CRM hygiene (e.g., the Vertex Labs duplicates)

**Why chosen:** the best balance of accuracy, interpretability, and feasibility given the data (29 months of history, 83 customers, a structured pipeline in CRM). For a quarter-length horizon this is the industry standard in B2B SaaS and cloud.


---
### 6.3 Executive Summary — for the business team

#### Context
NimbusCompute grows steadily: 472K/mo in January 2024 → 1,466K/mo in May 2026, ×3 over 29 months. 83 customers with usage in May (79 with `active` billing status), 5 products. This report is a forecast of consumption and cash collections for **June–August 2026** (as-of 2026-06-06).

---

#### Consumption Revenue forecast

| Month | Downside | **Base** | Upside |
|-------|--------|----------|--------|
| June 2026 | 1,581K | **1,639K** | 1,654K |
| July 2026 | 1,552K | **2,301K** | 2,400K |
| August 2026 | 1,524K | **2,409K** | 2,532K |
| **Jun–Aug total** | **4,657K** | **6,349K** | **6,586K** |

Base is ~+44% above the current run rate (1,466K/mo → ~2,116K/mo on average over the period). The main July–August driver is Vertex Labs ($690K/mo), which requires confirmation. The organic anchor is May (the last actual).

---

#### Cash Collections forecast

| Month | Downside | **Base** | Upside |
|-------|--------|----------|--------|
| June 2026 | 2,941K | **2,985K** | 2,985K |
| July 2026 | 1,420K | **7,703K** | 8,394K |
| August 2026 | 1,164K | **1,461K** | 1,471K |
| **Jun–Aug total** | **5,526K** | **12,148K** | **12,851K** |

**Vertex Labs — separately (conditional component):** base cash depends heavily on a single CRM record:

| Component | Base cash |
|---|---:|
| Core cash (excl. Vertex) | ~$5,938K |
| Conditional Vertex prepayment (OPP-1037, ×0.9) | +$6,211K |
| **Base total** | **$12,148K** |

**Important for Finance:** July cash in base/upside (~7.7–8.4M) is almost entirely Vertex annual-prepay ($6.9M, base ×0.9). Without it (downside) July = $1,420K: summer consumption is collected after the period, but the April–May receivables (AR as-of) hold the month up. Only the arrears share of organic enters the lag — the annual_prepay base generates no new cash. August base includes the OPP-1038 quarterly prepayment (~$210K). Critical for liquidity planning.

---

#### Three actions to complete before the end of June

1. **Sales → Vertex Labs** (OPP-1035/1036/1037): one deal or three? The answer drives both consumption and ~$6.2M of July cash
2. **Finance/Legal → OPP-1039**: confirm annual_prepay (1.8M in June). If monthly — June cash drops from 2,985K to ~1,185K
3. **Finance → AR**: work-out on the overdue receivables $1,442K (6–24 months); aging-specific recovery ~$88K/mo base


---
### 6.4 Open questions and flags

#### Critical (affect the forecast figures)

**🔴 Vertex Labs (OPP-1035, OPP-1036, OPP-1037 — account "Tesseract AI", ACC-0018)**
- Three deals on one account close on the same day (June 24) and start in the same month (July)
- In the model OPP-1035/1036 are excluded as duplicates, only OPP-1037 (690K/mo) is kept. Note: 410K + 230K = 640K ≠ 690K, i.e. OPP-1037 is not an exact sum of 1035+1036
- If OPP-1037 is just an umbrella record with no incremental of its own, the entire Vertex contribution must be removed; then July returns to ~1,552K (downside), and July base cash loses ~$6.21M of prepay (the June–August base drops to ~$5.94M)
- If OPP-1035/1036 are separate components, include them instead of OPP-1037, but not together with it
- **Needed from Sales**: confirmation of the deal structure by June 30

**🔴 OPP-1039 payment structure (account ACC-0058 "Tesseract AI Inc.", unmatched; likely = contract CON-1888, CUST-0058)**
- We assume annual_prepay → 1.8M in one shot in June (from committed_spend)
- If the contract turns out to be monthly → 150K/mo, and June cash drops from ~2,985K to ~1,185K
- The OPP-1039 ↔ CON-1888 link is an assumption (account unmatched); verify to avoid double counting
- **Needed from Finance/Legal**: confirmation of contract_type by June 25

**🟡 Churned customers with May usage**
- CUST-0011, CUST-0037, CUST-0012, CUST-0016 are flagged churned in the billing system (billing_customers) but have usage in May 2026
- Three possibilities: (a) a status-update lag — the customer is still active; (b) final consumption before offboarding; (c) a status error
- They are NOT included in the organic base (status ≠ active). If (a) — the base is understated
- **Needed from Account Management**: status as of June 2026

---

#### Important (affect analysis quality)

**🟡 Billing system change, April 2026**
- The data shows signs of non-standard behavior in April (invoice anomalies)
- April is used in the organic base (March–May average)
- **Needed from Finance**: confirmation that the April invoice data passed reconciliation

**🟡 network_egress concentration**
- 59% of revenue is egress. Two customers (CUST-0018, CUST-0079) are clear volume outliers
- The model assumes they continue at the same rate. There is no data on their contractual volume commitments
- **Needed from Sales**: committed usage contracts or pay-as-you-go?

**🟡 Overdue receivables 1,442K**
- Invoices 6–24 months overdue (~70% is debt older than 180 days). Recovery is computed **aging-specific**: 31–90d 50% / 91–180d 20% / >180d 5% (downside — half), base ~$262.9K for June–August (~$88K/mo base)
- The oldest buckets are nearly uncollectible — hence the low aggregate rate
- **Needed from AR**: status of each debtor (dispute, write-off, payment plan?)

---

#### Model assumptions (for reproducibility)

| Parameter | Value | Source |
|----------|----------|---------|
| As-of date | 2026-06-06 (last invoice_date) | invoices |
| Organic base | 1,460K/mo (May 2026, active clients — last actual) | monthly_usage + contracts |
| MoM growth (base / upside / downside) | +2% / +3% / −2% | 6-month trend with adjustment; downside — stress |
| Pipeline | recurring from start month, ramp-up by ramp_month, Renewal excluded | crm_opportunities + crm_opportunity_products |
| Lag W1 (t+1) / W2 (t+2) | 16.2% / 55.8% | Empirical from paid_date (service→cash, as-of censored) |
| Annual / quarterly prepay | annual: committed×12/term; quarterly: committed×3/term (fallback monthly×3), in the start month | crm_opportunities |
| Overdue recovery | aging-specific: 1-30d 80% / 31-90d 50% / 91-180d 20% / >180d 5% (downside — half); rates are scenario assumptions | Aging assumption; requires validation with AR |
| OPP-1035/1036 | Excluded as Vertex Labs duplicates | CRM analysis + sales policy May 2026 |
| Discounts | By the contract validity period (date-aware join) | contracts (contract_start..contract_end) |


---
## Appendix A — Usage detail by product

A detailed breakdown of usage by "customer × product" (GPU, Compute, Object Storage, Network Egress). Used during analysis to surface anomalies and concentration (e.g., the March GPU artifact for CUST-0059 and egress concentration on CUST-0018/0079), but it does not feed the forecast model directly. Moved out of Block 1 to keep the main narrative compact.


### A.1 GPU — usage by customer


In [50]:
df_gpu = con.sql("""
    SELECT month, customer_id, SUM(gpu_hours) AS gpu_hours
    FROM monthly_usage
    WHERE product IN ('gpu_a100', 'gpu_accelerated')
    GROUP BY month, customer_id
    ORDER BY month, customer_id
""").df()

fig = px.line(
    df_gpu, x="month", y="gpu_hours", color="customer_id",
    title="GPU hours per customer (gpu_a100 + gpu_accelerated combined)",
    labels={"month": "Month", "gpu_hours": "GPU hours", "customer_id": "Customer"}
)
fig.write_image("block1_gpu_usage_per_customer.png")
fig.show()

In [51]:
# Surface anomalies: top-10 rows by usage and the customer with a March 2026 spike
print("=== Top-10 months by GPU hours ===")
print(con.sql("""
    SELECT customer_id, month, SUM(gpu_hours) AS gpu_hours
    FROM monthly_usage
    WHERE product IN ('gpu_a100', 'gpu_accelerated')
    GROUP BY customer_id, month
    ORDER BY gpu_hours DESC
    LIMIT 10
""").df().to_string(index=False))

=== Top-10 months by GPU hours ===
customer_id      month  gpu_hours
  CUST-0018 2026-02-01   27883.47
  CUST-0018 2026-01-01   26171.40
  CUST-0018 2025-12-01   22425.32
  CUST-0018 2025-11-01   21204.71
  CUST-0018 2026-05-01   20880.72
  CUST-0033 2026-03-01   17859.46
  CUST-0018 2026-04-01   17494.52
  CUST-0018 2026-03-01   16739.89
  CUST-0018 2025-09-01    6497.17
  CUST-0022 2026-04-01    4228.43


In [52]:
# Customer with the highest GPU in March 2026 (the capacity_constraint month)
print("=== GPU usage in March 2026, TOP-5 ===")
con.sql("""
    SELECT customer_id, month, SUM(gpu_hours) AS gpu_hours
    FROM monthly_usage
    WHERE product IN ('gpu_a100', 'gpu_accelerated')
      AND month = '2026-03-01'
    GROUP BY customer_id, month
    ORDER BY gpu_hours DESC
    LIMIT 5
""").df()

=== GPU usage in March 2026, TOP-5 ===


,customer_id,month,gpu_hours
0,CUST-0033,2026-03-01,17859.46
1,CUST-0018,2026-03-01,16739.89
2,CUST-0022,2026-03-01,3012.17
3,CUST-0082,2026-03-01,2676.30
4,CUST-0058,2026-03-01,2529.92


In [53]:
# Customer with the most notable GPU growth — comparing H2 2025 vs H1
print("=== Customers with the largest GPU growth: comparing Oct-25–May-26 vs Jan-25–Sep-25 ===")
con.sql("""
    SELECT customer_id,
           ROUND(SUM(CASE WHEN month >= '2025-10-01' THEN gpu_hours ELSE 0 END) / 8, 0) AS avg_late_period,
           ROUND(SUM(CASE WHEN month < '2025-10-01'  THEN gpu_hours ELSE 0 END) / 9, 0) AS avg_early_period
    FROM monthly_usage
    WHERE product IN ('gpu_a100', 'gpu_accelerated')
    GROUP BY customer_id
    HAVING avg_late_period > 0
    ORDER BY (avg_late_period - avg_early_period) DESC
    LIMIT 5
""").df()

=== Customers with the largest GPU growth: comparing Oct-25–May-26 vs Jan-25–Sep-25 ===


,customer_id,avg_late_period,avg_early_period
0,CUST-0018,19511.0,6716.0
1,CUST-0033,4164.0,2766.0
2,CUST-0066,3348.0,2053.0
3,CUST-0027,868.0,333.0
4,CUST-0069,905.0,500.0


**Findings:**

The top-10 rows and the chart above reveal two non-standard patterns:
1. One customer with sharp, sustained GPU growth from late 2025 — a query over the post-October-2025 period confirms the growth.
2. A customer with an isolated spike strictly in March 2026 and normal usage before and after — coinciding with the `capacity_constraint` business event.

The first is likely organic growth (a new workload). The second coincides in time with the capacity constraint — without operational context it is impossible to cleanly separate real demand from the constraint effect; interpret with caution.

> **📋 Calculation note:** (1) Exclude March 2026 from the GPU baseline — a single customer's spike distorts the average. Baseline = April–May 2026. (2) The customer with sustained growth — count at the current April–May level.


### A.2 Compute — usage by customer


In [54]:
df_compute = con.sql("""
    SELECT month, customer_id, compute_hours
    FROM monthly_usage
    WHERE product = 'compute_standard'
    ORDER BY month, customer_id
""").df()

fig = px.line(
    df_compute, x="month", y="compute_hours", color="customer_id",
    title="Compute hours per customer (compute_standard)",
    labels={"month": "Month", "compute_hours": "Compute hours", "customer_id": "Customer"}
)
fig.write_image("block1_compute_usage_per_customer.png")
fig.show()

In [55]:
# Top customers by average compute usage for April–May 2026
con.sql("""
    SELECT customer_id,
           ROUND(AVG(compute_hours), 0) AS avg_compute_hours_apr_may
    FROM monthly_usage
    WHERE product = 'compute_standard'
      AND month IN ('2026-04-01', '2026-05-01')
    GROUP BY customer_id
    ORDER BY avg_compute_hours_apr_may DESC
    LIMIT 5
""").df()

,customer_id,avg_compute_hours_apr_may
0,CUST-0006,231713.0
1,CUST-0024,128223.0
2,CUST-0079,127793.0
3,CUST-0069,123889.0
4,CUST-0066,108980.0


**Findings:**

The chart and the top-5 table show one clearly dominant customer — an order of magnitude above the rest. A few customers with moderate growth. Most have stable, small usage.

> **📋 Calculation note:** The top compute customer carries a disproportionate weight in the baseline. When computing the trend, check whether a single change at this customer could distort the entire organic forecast.


### A.3 Object Storage — usage by customer


In [56]:
df_storage = con.sql("""
    SELECT month, customer_id, storage_tb_avg
    FROM monthly_usage
    WHERE product = 'object_storage'
    ORDER BY month, customer_id
""").df()

fig = px.line(
    df_storage, x="month", y="storage_tb_avg", color="customer_id",
    title="Object storage (TB avg) per customer",
    labels={"month": "Month", "storage_tb_avg": "Storage TB avg", "customer_id": "Customer"}
)
fig.write_image("block1_storage_usage_per_customer.png")
fig.show()

**Findings:**

Object storage is the most "predictable" product: customer usage is stable and grows smoothly without sharp spikes. A few customers stand out with significantly larger volumes.

> **📋 Calculation note:** Object storage is well suited to trend forecasting — low volatility. Account for the January 2026 price cut when computing historical revenue.


### A.4 Network Egress — usage by customer


In [57]:
df_egress = con.sql("""
    SELECT month, customer_id, egress_tb
    FROM monthly_usage
    WHERE product = 'network_egress'
    ORDER BY month, customer_id
""").df()

fig = px.line(
    df_egress, x="month", y="egress_tb", color="customer_id",
    title="Network egress (TB) per customer",
    labels={"month": "Month", "egress_tb": "Egress TB", "customer_id": "Customer"}
)
fig.write_image("block1_egress_usage_per_customer.png")
fig.show()

**Findings:**

Network egress is the largest revenue source by volume. The chart clearly shows 2–3 customers with far more traffic than the rest, creating high concentration.

> **📋 Calculation note:** High concentration: a few large customers provide a significant share of egress revenue. The departure of any one of them would have a noticeable effect.
